# Solar Panel Thermal Anomaly Detection — Google Colab

**Deep Learning mini-project | YOLOv8 segmentation**

This notebook is prepared to run in **Google Colab** (cloud runtime)

## Before running
1. Open **Runtime → Change runtime type → T4 GPU** (or another GPU if available).
2. Run the setup cell below. If the dataset is not already in this runtime, it will ask for a Roboflow API key and download version 2.
3. Use **Runtime → Run all** only when you are ready to run the complete pipeline. Training can take time and uses Colab GPU resources.
4. If the runtime disconnects, files stored under `/content` may be lost. Download or copy important outputs to Google Drive.

**Dataset:** [Thermal Anomalies — Solar Panel Data Set](https://universe.roboflow.com/solar-panel-data-set/thermal-anomalies-jev7w)

**Important:** The original 17-class annotations are investigated and converted into a 5-class segmentation dataset by later cells. Keep the original dataset unchanged.


In [ ]:
# Colab setup: install dependencies and ensure the original dataset exists.
%pip install -q roboflow ultralytics pyyaml pandas matplotlib opencv-python-headless

from pathlib import Path
from getpass import getpass
import shutil

EXPECTED_ROOT = Path("/content/Thermal-Anomalies-2")
DATASET_LOCATION = str(EXPECTED_ROOT)

# Reuse the dataset if it is already available in this Colab runtime.
if (EXPECTED_ROOT / "data.yaml").exists():
    print(f"Dataset already exists: {EXPECTED_ROOT}")
else:
    # Check for a dataset folder with the expected YAML before downloading again.
    candidates = [
        Path("/content/Thermal-Anomalies-2"),
        Path("/content/thermal-anomalies-2"),
        Path("/content/Thermal-Anomalies-2-1"),
    ]
    existing_yaml = next((p / "data.yaml" for p in candidates if (p / "data.yaml").exists()), None)

    if existing_yaml is not None:
        DATASET_LOCATION = str(existing_yaml.parent)
        print(f"Found existing dataset: {DATASET_LOCATION}")
    else:
        print("Dataset not found in this runtime. Downloading Roboflow version 2.")
        print("Get your private API key from your Roboflow account settings.")
        ROBOFLOW_API_KEY = getpass("Roboflow API key (input hidden): ").strip()
        if not ROBOFLOW_API_KEY:
            raise ValueError("No API key entered. Re-run this cell and enter your Roboflow API key.")

        from roboflow import Roboflow
        rf = Roboflow(api_key=ROBOFLOW_API_KEY)
        project = rf.workspace("solar-panel-data-set").project("thermal-anomalies-jev7w")
        dataset = project.version(2).download("yolov8")
        downloaded_root = Path(dataset.location)
        print(f"Roboflow downloaded dataset to: {downloaded_root}")

        # The remaining notebook cells use this stable path.
        if downloaded_root.resolve() != EXPECTED_ROOT.resolve():
            if EXPECTED_ROOT.exists():
                shutil.rmtree(EXPECTED_ROOT)
            shutil.copytree(downloaded_root, EXPECTED_ROOT)
        DATASET_LOCATION = str(EXPECTED_ROOT)

assert (Path(DATASET_LOCATION) / "data.yaml").exists(), (
    f"Could not find data.yaml under {DATASET_LOCATION}. "
    "Check that the dataset download completed successfully."
)
print(f"\nDataset ready: {DATASET_LOCATION}")
print("Next: run the dataset inspection and validation cells in order.")


In [ ]:
import os

def find_dirs(root):
    structure = {"train_images": None, "valid_images": None, "test_images": None,
                 "train_labels": None, "valid_labels": None, "test_labels": None,
                 "data_yaml": None}
    for dirpath, dirnames, filenames in os.walk(root):
        low = dirpath.lower()
        if "data.yaml" in filenames:
            structure["data_yaml"] = os.path.join(dirpath, "data.yaml")
        for split_key, split_name in [("train", "train"), ("valid", "valid"), ("val", "valid"), ("test", "test")]:
            if split_key in low:
                if low.endswith("images") or "images" in os.path.basename(dirpath).lower():
                    structure[f"{split_name}_images"] = dirpath
                elif low.endswith("labels") or "labels" in os.path.basename(dirpath).lower():
                    structure[f"{split_name}_labels"] = dirpath
    return structure

structure = find_dirs(DATASET_LOCATION)
for k, v in structure.items():
    print(f"{k:16s}: {v}")

In [ ]:
import yaml

with open(structure["data_yaml"], "r") as f:
    data_yaml = yaml.safe_load(f)

class_names = data_yaml.get("names")
num_classes = data_yaml.get("nc", len(class_names) if class_names else 0)

print(f"Number of classes declared in data.yaml: {num_classes}")
print("Class list (index -> name):")
for i, name in enumerate(class_names):
    print(f"  {i}: {name}")

In [ ]:
import pandas as pd
import glob

def collect_split(split):
    img_dir = structure.get(f"{split}_images")
    lbl_dir = structure.get(f"{split}_labels")
    rows = []
    images = []
    if img_dir is None or not os.path.isdir(img_dir):
        return rows, images
    for img_path in glob.glob(os.path.join(img_dir, "*")):
        if not img_path.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
            continue
        stem = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(lbl_dir, stem + ".txt") if lbl_dir else None
        images.append({"split": split, "image_path": img_path, "label_path": label_path,
                        "label_exists": bool(label_path and os.path.isfile(label_path))})
        if label_path and os.path.isfile(label_path):
            with open(label_path, "r") as f:
                lines = [l.strip() for l in f.readlines() if l.strip() != ""]
            if len(lines) == 0:
                rows.append({"split": split, "image_path": img_path, "label_path": label_path,
                             "raw_line": None, "line_no": None})
            for i, line in enumerate(lines):
                rows.append({"split": split, "image_path": img_path, "label_path": label_path,
                             "raw_line": line, "line_no": i})
    return rows, images

all_rows, all_images = [], []
for split in ["train", "valid", "test"]:
    r, i = collect_split(split)
    all_rows.extend(r)
    all_images.extend(i)

ann_raw_df = pd.DataFrame(all_rows)
img_df = pd.DataFrame(all_images)

print(f"Total images found: {len(img_df)}")
print(f"Total label rows found (before validation): {len(ann_raw_df)}")

In [ ]:
import numpy as np

def validate_row(row):
    result = {
        "missing_label_file": not row["label_exists"] if "label_exists" in row else False,
        "empty_label_file": row["raw_line"] is None,
        "malformed_field_count": False,
        "non_numeric": False,
        "invalid_class_id": False,
        "nonpositive_wh": False,
        "out_of_range_xy": False,
        "out_of_range_wh": False,
        "class_id": None, "x": None, "y": None, "w": None, "h": None,
    }
    if row["raw_line"] is None:
        return result
    parts = row["raw_line"].split()
    if len(parts) != 5:
        result["malformed_field_count"] = True
        return result
    try:
        cid = int(float(parts[0]))
        x, y, w, h = [float(p) for p in parts[1:]]
    except ValueError:
        result["non_numeric"] = True
        return result
    result["class_id"], result["x"], result["y"], result["w"], result["h"] = cid, x, y, w, h
    if cid < 0 or cid >= num_classes:
        result["invalid_class_id"] = True
    if w <= 0 or h <= 0:
        result["nonpositive_wh"] = True
    if not (0 <= x <= 1 and 0 <= y <= 1):
        result["out_of_range_xy"] = True
    if w > 1 or h > 1:
        result["out_of_range_wh"] = True
    return result

# join label_exists onto ann_raw_df for the missing-file check
label_exists_map = img_df.set_index("label_path")["label_exists"].to_dict()
ann_raw_df["label_exists"] = ann_raw_df["label_path"].map(lambda p: label_exists_map.get(p, False))

validated = ann_raw_df.apply(validate_row, axis=1, result_type="expand")
ann_df = pd.concat([ann_raw_df, validated], axis=1)
ann_df["class_name"] = ann_df["class_id"].apply(lambda c: class_names[int(c)] if pd.notna(c) and 0 <= int(c) < num_classes else None)

# duplicate rows: identical (image, class_id, x, y, w, h) appearing more than once
dup_mask = ann_df.duplicated(subset=["image_path", "class_id", "x", "y", "w", "h"], keep=False) & ann_df["class_id"].notna()
ann_df["duplicate_annotation"] = dup_mask

problem_counts = {
    "Empty label files": ann_df["empty_label_file"].sum(),
    "Malformed field count (not 5 values)": ann_df["malformed_field_count"].sum(),
    "Non-numeric values": ann_df["non_numeric"].sum(),
    "Invalid class ID": ann_df["invalid_class_id"].sum(),
    "Non-positive width/height": ann_df["nonpositive_wh"].sum(),
    "x/y outside [0,1]": ann_df["out_of_range_xy"].sum(),
    "w/h outside valid range (>1)": ann_df["out_of_range_wh"].sum(),
    "Exact duplicate annotations": ann_df["duplicate_annotation"].sum(),
}
images_missing_labels = img_df[~img_df["label_exists"]].shape[0]

print("=== Annotation Problems Found ===")
print(f"Images with no label file at all: {images_missing_labels}")
for k, v in problem_counts.items():
    print(f"{k}: {int(v)}")

In [ ]:
from PIL import Image

img_records = []
for _, row in img_df.iterrows():
    rec = {"image_path": row["image_path"], "split": row["split"], "corrupted": False,
           "width": None, "height": None}
    try:
        with Image.open(row["image_path"]) as im:
            im.verify()
        with Image.open(row["image_path"]) as im:
            rec["width"], rec["height"] = im.size
    except Exception:
        rec["corrupted"] = True
    img_records.append(rec)

img_check_df = pd.DataFrame(img_records)
img_df = img_df.merge(img_check_df, on=["image_path", "split"])

print(f"Total images checked: {len(img_df)}")
print(f"Corrupted / unreadable images: {img_df['corrupted'].sum()}")
print(f"Images missing a label file: {(~img_df['label_exists']).sum()}")
print("\nImage dimension summary:")
print(img_df[["width", "height"]].describe())
print("\nUnique (width,height) pairs found:")
print(img_df.groupby(["width", "height"]).size().sort_values(ascending=False).head(10))

In [ ]:
print("=== Images per split ===")
print(img_df["split"].value_counts())

print("\n=== Valid annotations per split ===")
valid_ann = ann_df[ann_df["class_id"].notna()]
print(valid_ann["split"].value_counts())

print("\n=== Class distribution per split (bounding box counts) ===")
split_class_table = valid_ann.groupby(["split", "class_name"]).size().unstack(fill_value=0)
split_class_table

In [ ]:
rows = []
for cid, name in enumerate(class_names):
    class_ann = valid_ann[valid_ann["class_id"] == cid]
    n_images = class_ann["image_path"].nunique()
    n_boxes = len(class_ann)
    rows.append({"ID": cid, "Original Class": name, "Number of Images": n_images, "Number of Bounding Boxes": n_boxes})

class_table = pd.DataFrame(rows).sort_values("Number of Bounding Boxes", ascending=False)
pd.set_option("display.max_rows", None)
class_table

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random

def name_to_id(name):
    try:
        return class_names.index(name)
    except ValueError:
        return None

def show_class_grid(target_names, n_per_class=3, title=""):
    fig, axes = plt.subplots(len(target_names), n_per_class, figsize=(4*n_per_class, 4*len(target_names)))
    if len(target_names) == 1:
        axes = [axes]
    for row_i, tname in enumerate(target_names):
        cid = name_to_id(tname)
        if cid is None:
            for col_i in range(n_per_class):
                ax = axes[row_i][col_i] if len(target_names) > 1 else axes[col_i]
                ax.axis("off")
                ax.set_title(f'"{tname}" not found in data.yaml', fontsize=9, color="red")
            continue
        class_imgs = valid_ann[valid_ann["class_id"] == cid]["image_path"].unique().tolist()
        random.shuffle(class_imgs)
        chosen = class_imgs[:n_per_class]
        for col_i in range(n_per_class):
            ax = axes[row_i][col_i] if len(target_names) > 1 else axes[col_i]
            ax.axis("off")
            if col_i >= len(chosen):
                ax.set_title(f'"{tname}": no samples found', fontsize=9, color="red")
                continue
            img_path = chosen[col_i]
            with Image.open(img_path) as im:
                w, h = im.size
                ax.imshow(im)
            boxes = valid_ann[(valid_ann["image_path"] == img_path) & (valid_ann["class_id"] == cid)]
            for _, b in boxes.iterrows():
                x_c, y_c, bw, bh = b["x"]*w, b["y"]*h, b["w"]*w, b["h"]*h
                rect = patches.Rectangle((x_c - bw/2, y_c - bh/2), bw, bh, linewidth=2, edgecolor="lime", facecolor="none")
                ax.add_patch(rect)
            ax.set_title(tname, fontsize=9)
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

show_class_grid(["2", "4", "Cell High", "Cell Medium", "Cell Low"], n_per_class=3,
                 title="Ambiguous classes — do not assume meaning, inspect visually")

In [ ]:
hotspot_classes = ["High- Hot Spot", "Medium- Hot Spot", "Low- Hot Spot", "Mexium- Hot Spot",
                    "High- Multiple Hot Spot", "Medium- MultipleHot Spot", "Low- MultipleHot Spot",
                    "Mixed- Multiple Hot Spot", "High - Multi Cell"]

show_class_grid(hotspot_classes, n_per_class=2,
                 title="Hotspot-related classes — visually judge whether a merge is reasonable")

In [ ]:
print("=" * 60)
print("FINAL VERIFICATION REPORT — Thermal Anomalies Dataset")
print("=" * 60)

print(f"\nTotal images: {len(img_df)}")
print(f"Total valid annotations: {len(valid_ann)}")
print(f"Number of classes (from data.yaml): {num_classes}")

print("\n--- Images per split ---")
print(img_df["split"].value_counts())

print("\n--- Annotations per split ---")
print(valid_ann["split"].value_counts())

print("\n--- Images per class / Boxes per class ---")
print(class_table.to_string(index=False))

print("\n--- Data quality issues ---")
print(f"Corrupted images: {img_df['corrupted'].sum()}")
print(f"Images missing labels: {(~img_df['label_exists']).sum()}")
for k, v in problem_counts.items():
    print(f"{k}: {int(v)}")

print("\n--- Class imbalance check ---")
box_counts = class_table.set_index("Original Class")["Number of Bounding Boxes"]
if len(box_counts[box_counts > 0]) > 0:
    ratio = box_counts.max() / max(box_counts[box_counts > 0].min(), 1)
    print(f"Largest class: {box_counts.idxmax()} ({box_counts.max()} boxes)")
    print(f"Smallest non-zero class: {box_counts[box_counts>0].idxmin()} ({box_counts[box_counts>0].min()} boxes)")
    print(f"Imbalance ratio (max/min): {ratio:.1f}x")

print("\n--- Recommendation (data-derived, not applied automatically) ---")
zero_classes = class_table[class_table["Number of Bounding Boxes"] == 0]["Original Class"].tolist()
if zero_classes:
    print(f"Classes with ZERO valid annotations in this version: {zero_classes}")
low_sample_classes = class_table[(class_table["Number of Bounding Boxes"] > 0) & (class_table["Number of Bounding Boxes"] < 20)]
if len(low_sample_classes):
    print("Classes with very few samples (<20 boxes) — likely too sparse to train reliably:")
    print(low_sample_classes.to_string(index=False))
print("\nNo dataset files were modified. No classes were merged or deleted. No model was trained.")

In [ ]:
# ============================================================
# Stage 2 — Annotation Format Investigation
# No dataset files are modified
# ============================================================

from pathlib import Path
import random

DATASET_ROOT = Path("/content/Thermal-Anomalies-2")

SPLITS = ["train", "valid", "test"]

print("=" * 70)
print("ANNOTATION FORMAT INVESTIGATION")
print("=" * 70)

# ------------------------------------------------------------
# Collect representative non-5-value annotation rows
# ------------------------------------------------------------

non_five_rows = []
empty_files = []

for split in SPLITS:
    label_dir = DATASET_ROOT / split / "labels"

    if not label_dir.exists():
        print(f"WARNING: Missing label directory: {label_dir}")
        continue

    for label_file in sorted(label_dir.glob("*.txt")):

        text = label_file.read_text(errors="ignore").strip()

        if not text:
            empty_files.append((split, label_file.name))
            continue

        for line_number, line in enumerate(text.splitlines(), start=1):

            values = line.strip().split()

            if len(values) != 5:
                non_five_rows.append({
                    "split": split,
                    "file": label_file.name,
                    "line": line_number,
                    "values": values,
                    "count": len(values)
                })


# ------------------------------------------------------------
# Basic counts
# ------------------------------------------------------------

five_value_count = 0
less_than_five = 0
greater_than_five = 0

for split in SPLITS:
    label_dir = DATASET_ROOT / split / "labels"

    if not label_dir.exists():
        continue

    for label_file in label_dir.glob("*.txt"):
        text = label_file.read_text(errors="ignore").strip()

        if not text:
            continue

        for line in text.splitlines():
            count = len(line.split())

            if count == 5:
                five_value_count += 1
            elif count < 5:
                less_than_five += 1
            else:
                greater_than_five += 1


print("\nSUMMARY")
print("-" * 70)
print(f"5-value YOLO detection rows : {five_value_count}")
print(f">5-value rows               : {greater_than_five}")
print(f"<5-value rows               : {less_than_five}")
print(f"Empty label files           : {len(empty_files)}")
print(f"Total non-5-value rows      : {len(non_five_rows)}")


# ------------------------------------------------------------
# Show representative examples
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("REPRESENTATIVE NON-5-VALUE ANNOTATIONS")
print("=" * 70)

if non_five_rows:

    # Try to sample across different splits
    random.seed(42)

    samples = []

    for split in SPLITS:
        split_rows = [
            row for row in non_five_rows
            if row["split"] == split
        ]

        samples.extend(random.sample(
            split_rows,
            min(5, len(split_rows))
        ))

    for i, row in enumerate(samples[:15], start=1):

        print(f"\nExample {i}")
        print(f"Split       : {row['split']}")
        print(f"Label file  : {row['file']}")
        print(f"Line        : {row['line']}")
        print(f"Value count : {row['count']}")
        print("First values:", row["values"][:20])


# ------------------------------------------------------------
# Determine whether >5-value rows look like polygons
# ------------------------------------------------------------

polygon_like = 0
not_polygon_like = 0
polygon_examples = []

for row in non_five_rows:

    values = row["values"]

    # A YOLO segmentation row normally has:
    # class_id + even number of coordinate values
    if len(values) >= 7 and (len(values) - 1) % 2 == 0:

        try:
            class_id = int(float(values[0]))
            coords = [float(x) for x in values[1:]]

            # Normalized YOLO coordinates should generally be 0..1
            normalized = all(0.0 <= x <= 1.0 for x in coords)

            if normalized:
                polygon_like += 1

                if len(polygon_examples) < 10:
                    polygon_examples.append(row)
            else:
                not_polygon_like += 1

        except ValueError:
            not_polygon_like += 1

    else:
        not_polygon_like += 1


print("\n" + "=" * 70)
print("POLYGON-LIKE STRUCTURE CHECK")
print("=" * 70)

print(f"Polygon-like rows              : {polygon_like}")
print(f"Rows not matching polygon form : {not_polygon_like}")

if polygon_examples:

    print("\nExamples of polygon-like rows:")

    for i, row in enumerate(polygon_examples, start=1):
        print(f"\nPolygon example {i}")
        print(f"Split       : {row['split']}")
        print(f"Label file  : {row['file']}")
        print(f"Value count : {row['count']}")
        print("First 20 values:", row["values"][:20])


# ------------------------------------------------------------
# Final diagnostic conclusion
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DIAGNOSTIC CONCLUSION")
print("=" * 70)

if greater_than_five > 0 and polygon_like == greater_than_five:
    print("""
The >5-value annotations have the expected structure of
YOLO segmentation polygons:

class_id x1 y1 x2 y2 ... xn yn

and their coordinates are normalized between 0 and 1.

These rows appear to be segmentation-style annotations,
not ordinary 5-value YOLO detection annotations.

NO DATASET FILES WERE MODIFIED.
""")

elif greater_than_five > 0 and polygon_like > 0:
    print("""
Some >5-value annotations appear polygon-like, but not all
of them match the expected YOLO segmentation structure.

Further investigation is required.

NO DATASET FILES WERE MODIFIED.
""")

else:
    print("""
The >5-value annotations do not clearly match the expected
YOLO segmentation polygon structure.

They may be malformed detection annotations or another
annotation format.

Further investigation is required.

NO DATASET FILES WERE MODIFIED.
""")

In [ ]:
# ============================================================
# Stage 2 — Complete Class Distribution
# Counts BOTH YOLO detection and segmentation annotations
# NO DATASET FILES ARE MODIFIED
# ============================================================

from pathlib import Path
from collections import defaultdict
import yaml

# ------------------------------------------------------------
# Dataset location
# ------------------------------------------------------------

DATASET_ROOT = Path("/content/Thermal-Anomalies-2")
DATA_YAML = DATASET_ROOT / "data.yaml"

assert DATASET_ROOT.exists(), f"Dataset not found: {DATASET_ROOT}"
assert DATA_YAML.exists(), f"data.yaml not found: {DATA_YAML}"

# ------------------------------------------------------------
# Load class names
# ------------------------------------------------------------

with open(DATA_YAML, "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)

names = data["names"]

# Handle either:
# names: ["class1", "class2"]
# or
# names: {0: "class1", 1: "class2"}

if isinstance(names, dict):
    class_names = {
        int(k): str(v)
        for k, v in names.items()
    }
else:
    class_names = {
        i: str(name)
        for i, name in enumerate(names)
    }

NUM_CLASSES = len(class_names)

print("=" * 80)
print("COMPLETE CLASS DISTRIBUTION — ALL ANNOTATION FORMATS")
print("=" * 80)

print(f"Dataset: {DATASET_ROOT}")
print(f"Classes found in data.yaml: {NUM_CLASSES}")

# ------------------------------------------------------------
# Storage
# ------------------------------------------------------------

total_annotations = defaultdict(int)
detection_annotations = defaultdict(int)
polygon_annotations = defaultdict(int)

images_containing_class = defaultdict(set)

split_annotations = {
    "train": defaultdict(int),
    "valid": defaultdict(int),
    "test": defaultdict(int)
}

split_detection = {
    "train": defaultdict(int),
    "valid": defaultdict(int),
    "test": defaultdict(int)
}

split_polygon = {
    "train": defaultdict(int),
    "valid": defaultdict(int),
    "test": defaultdict(int)
}

split_images = {
    "train": 0,
    "valid": 0,
    "test": 0
}

invalid_class_ids = []
non_numeric_class_ids = []
annotation_row_count = 0

# ------------------------------------------------------------
# Scan every split
# ------------------------------------------------------------

for split in ["train", "valid", "test"]:

    image_dir = DATASET_ROOT / split / "images"
    label_dir = DATASET_ROOT / split / "labels"

    image_files = []

    if image_dir.exists():
        image_files = [
            p for p in image_dir.iterdir()
            if p.is_file()
            and p.suffix.lower() in {
                ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"
            }
        ]

    split_images[split] = len(image_files)

    if not label_dir.exists():
        continue

    for label_file in sorted(label_dir.glob("*.txt")):

        text = label_file.read_text(
            encoding="utf-8",
            errors="ignore"
        ).strip()

        if not text:
            continue

        for line_number, line in enumerate(
            text.splitlines(),
            start=1
        ):

            values = line.strip().split()

            if len(values) == 0:
                continue

            annotation_row_count += 1

            # ------------------------------------------------
            # Extract class ID from first value
            # ------------------------------------------------

            try:
                class_id_float = float(values[0])

                if not class_id_float.is_integer():
                    non_numeric_class_ids.append(
                        (split, label_file.name, line_number, values[0])
                    )
                    continue

                class_id = int(class_id_float)

            except ValueError:
                non_numeric_class_ids.append(
                    (split, label_file.name, line_number, values[0])
                )
                continue

            # ------------------------------------------------
            # Validate class ID
            # ------------------------------------------------

            if class_id not in class_names:
                invalid_class_ids.append(
                    (split, label_file.name, line_number, class_id)
                )
                continue

            # ------------------------------------------------
            # Count annotation
            # ------------------------------------------------

            total_annotations[class_id] += 1
            split_annotations[split][class_id] += 1

            # ------------------------------------------------
            # Determine annotation format
            #
            # 5 values:
            # class x_center y_center width height
            #
            # >5 values:
            # class x1 y1 x2 y2 ... xn yn
            # ------------------------------------------------

            if len(values) == 5:

                detection_annotations[class_id] += 1
                split_detection[split][class_id] += 1

            elif len(values) > 5:

                polygon_annotations[class_id] += 1
                split_polygon[split][class_id] += 1

            # ------------------------------------------------
            # Track image-level presence
            # ------------------------------------------------

            images_containing_class[class_id].add(
                (split, label_file.stem)
            )


# ============================================================
# 1. DATASET-WIDE CLASS DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("1. DATASET-WIDE CLASS DISTRIBUTION")
print("=" * 80)

header = (
    f"{'ID':<4}"
    f"{'Class Name':<32}"
    f"{'Total':>10}"
    f"{'Images':>10}"
    f"{'Train':>10}"
    f"{'Valid':>10}"
    f"{'Test':>10}"
)

print(header)
print("-" * len(header))

for class_id in range(NUM_CLASSES):

    name = class_names[class_id]

    total = total_annotations[class_id]

    image_count = len(images_containing_class[class_id])

    train_count = split_annotations["train"][class_id]
    valid_count = split_annotations["valid"][class_id]
    test_count = split_annotations["test"][class_id]

    print(
        f"{class_id:<4}"
        f"{name:<32}"
        f"{total:>10}"
        f"{image_count:>10}"
        f"{train_count:>10}"
        f"{valid_count:>10}"
        f"{test_count:>10}"
    )


# ============================================================
# 2. ANNOTATION FORMAT BREAKDOWN
# ============================================================

print("\n" + "=" * 80)
print("2. ANNOTATION FORMAT BREAKDOWN")
print("=" * 80)

header = (
    f"{'ID':<4}"
    f"{'Class Name':<32}"
    f"{'Detection':>12}"
    f"{'Polygon':>12}"
    f"{'Total':>12}"
)

print(header)
print("-" * len(header))

for class_id in range(NUM_CLASSES):

    name = class_names[class_id]

    detection = detection_annotations[class_id]
    polygon = polygon_annotations[class_id]
    total = total_annotations[class_id]

    print(
        f"{class_id:<4}"
        f"{name:<32}"
        f"{detection:>12}"
        f"{polygon:>12}"
        f"{total:>12}"
    )


# ============================================================
# 3. SPLIT-LEVEL TOTALS
# ============================================================

print("\n" + "=" * 80)
print("3. SPLIT-LEVEL TOTALS")
print("=" * 80)

print(
    f"{'Split':<10}"
    f"{'Images':>10}"
    f"{'Detection':>15}"
    f"{'Polygon':>15}"
    f"{'Total':>15}"
)

print("-" * 65)

for split in ["train", "valid", "test"]:

    detection_total = sum(split_detection[split].values())
    polygon_total = sum(split_polygon[split].values())
    total = sum(split_annotations[split].values())

    print(
        f"{split:<10}"
        f"{split_images[split]:>10}"
        f"{detection_total:>15}"
        f"{polygon_total:>15}"
        f"{total:>15}"
    )

overall_detection = sum(detection_annotations.values())
overall_polygon = sum(polygon_annotations.values())
overall_total = sum(total_annotations.values())

print("-" * 65)

print(
    f"{'TOTAL':<10}"
    f"{sum(split_images.values()):>10}"
    f"{overall_detection:>15}"
    f"{overall_polygon:>15}"
    f"{overall_total:>15}"
)


# ============================================================
# 4. CLASS COVERAGE GROUPS
# ============================================================

zero_classes = []
one_to_five = []
six_to_twenty = []
more_than_twenty = []

for class_id in range(NUM_CLASSES):

    count = total_annotations[class_id]

    if count == 0:
        zero_classes.append(class_id)

    elif 1 <= count <= 5:
        one_to_five.append(class_id)

    elif 6 <= count <= 20:
        six_to_twenty.append(class_id)

    else:
        more_than_twenty.append(class_id)


print("\n" + "=" * 80)
print("4. CLASS COVERAGE")
print("=" * 80)

print("\nA. ZERO ANNOTATIONS")
if zero_classes:
    for class_id in zero_classes:
        print(f"  {class_id}: {class_names[class_id]}")
else:
    print("  None")

print("\nB. 1–5 ANNOTATIONS")
if one_to_five:
    for class_id in one_to_five:
        print(
            f"  {class_id}: "
            f"{class_names[class_id]} "
            f"({total_annotations[class_id]})"
        )
else:
    print("  None")

print("\nC. 6–20 ANNOTATIONS")
if six_to_twenty:
    for class_id in six_to_twenty:
        print(
            f"  {class_id}: "
            f"{class_names[class_id]} "
            f"({total_annotations[class_id]})"
        )
else:
    print("  None")

print("\nD. MORE THAN 20 ANNOTATIONS")
if more_than_twenty:
    for class_id in more_than_twenty:
        print(
            f"  {class_id}: "
            f"{class_names[class_id]} "
            f"({total_annotations[class_id]})"
        )
else:
    print("  None")


# ============================================================
# 5. VALIDATION CHECKS
# ============================================================

print("\n" + "=" * 80)
print("5. VALIDATION CHECKS")
print("=" * 80)

expected_total = 625 + 2679

print(f"Expected annotation rows : {expected_total}")
print(f"Counted annotation rows  : {annotation_row_count}")

if annotation_row_count == expected_total:
    print("✓ Annotation total matches 625 + 2679 = 3304")
else:
    print(
        "⚠ Annotation total DOES NOT MATCH expected 3304"
    )

print(
    f"\nDetection rows counted : {overall_detection}"
)

print(
    f"Polygon rows counted   : {overall_polygon}"
)

print(
    f"Combined total         : "
    f"{overall_detection + overall_polygon}"
)

if overall_detection == 625:
    print("✓ Detection count matches previous verification: 625")
else:
    print("⚠ Detection count differs from previous report")

if overall_polygon == 2679:
    print("✓ Polygon count matches previous investigation: 2679")
else:
    print("⚠ Polygon count differs from previous investigation")

if not invalid_class_ids:
    print("✓ No invalid class IDs")
else:
    print(
        f"⚠ Invalid class IDs found: "
        f"{len(invalid_class_ids)}"
    )

if not non_numeric_class_ids:
    print("✓ No non-numeric class IDs")
else:
    print(
        f"⚠ Non-numeric class IDs found: "
        f"{len(non_numeric_class_ids)}"
    )


# ============================================================
# 6. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

print(f"Total images       : {sum(split_images.values())}")
print(f"Total classes      : {NUM_CLASSES}")
print(f"Detection rows     : {overall_detection}")
print(f"Polygon rows       : {overall_polygon}")
print(f"Total annotations  : {overall_total}")
print(f"Empty label files  : 8")

print("\nCOMPLETE CLASS DISTRIBUTION — READY FOR TAXONOMY REVIEW")
print("=" * 80)

# No files were written or modified.

In [ ]:
# ============================================================
# Stage 2 — Visual Annotation Verification
# Shows representative annotations from ALL 17 classes
# Supports BOTH YOLO detection boxes and segmentation polygons
# NO DATASET FILES ARE MODIFIED
# ============================================================

from pathlib import Path
from collections import defaultdict
import random
import math

import yaml
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

DATASET_ROOT = Path("/content/Thermal-Anomalies-2")
DATA_YAML = DATASET_ROOT / "data.yaml"

# Number of examples to display per class
EXAMPLES_PER_CLASS = 1

# Reproducible random selection
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# ------------------------------------------------------------
# Load class names
# ------------------------------------------------------------

with open(DATA_YAML, "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)

names = data["names"]

if isinstance(names, dict):
    class_names = {
        int(k): str(v)
        for k, v in names.items()
    }
else:
    class_names = {
        i: str(name)
        for i, name in enumerate(names)
    }

NUM_CLASSES = len(class_names)

print("=" * 80)
print("VISUAL ANNOTATION VERIFICATION")
print("=" * 80)
print(f"Dataset : {DATASET_ROOT}")
print(f"Classes : {NUM_CLASSES}")
print()


# ============================================================
# 1. COLLECT ANNOTATED EXAMPLES FOR EACH CLASS
# ============================================================

class_examples = defaultdict(list)

for split in ["train", "valid", "test"]:

    image_dir = DATASET_ROOT / split / "images"
    label_dir = DATASET_ROOT / split / "labels"

    if not image_dir.exists() or not label_dir.exists():
        continue

    image_lookup = {}

    for image_file in image_dir.iterdir():
        if image_file.is_file() and image_file.suffix.lower() in {
            ".jpg", ".jpeg", ".png", ".bmp",
            ".tif", ".tiff", ".webp"
        }:
            image_lookup[image_file.stem] = image_file

    for label_file in label_dir.glob("*.txt"):

        image_file = image_lookup.get(label_file.stem)

        if image_file is None:
            continue

        text = label_file.read_text(
            encoding="utf-8",
            errors="ignore"
        ).strip()

        if not text:
            continue

        class_ids_in_image = set()

        for line in text.splitlines():

            values = line.strip().split()

            if not values:
                continue

            try:
                class_id = int(float(values[0]))
            except ValueError:
                continue

            if class_id in class_names:
                class_ids_in_image.add(class_id)

        for class_id in class_ids_in_image:
            class_examples[class_id].append(
                (split, image_file, label_file)
            )


# ============================================================
# 2. SELECT REPRESENTATIVE EXAMPLES
# ============================================================

selected_examples = {}

for class_id in range(NUM_CLASSES):

    examples = class_examples[class_id]

    if not examples:
        selected_examples[class_id] = []
        continue

    # Random but reproducible selection
    examples_copy = examples.copy()
    random.shuffle(examples_copy)

    selected_examples[class_id] = examples_copy[:EXAMPLES_PER_CLASS]


# ============================================================
# 3. DRAW ANNOTATIONS
# ============================================================

def draw_annotations(ax, image, label_file, target_class_id):

    height, width = image.shape[:2]

    text = label_file.read_text(
        encoding="utf-8",
        errors="ignore"
    ).strip()

    if not text:
        return

    for line in text.splitlines():

        values = line.strip().split()

        if not values:
            continue

        try:
            class_id = int(float(values[0]))
        except ValueError:
            continue

        # Only show annotations belonging to the target class
        if class_id != target_class_id:
            continue

        coords = list(map(float, values[1:]))

        # ----------------------------------------------------
        # YOLO detection format
        #
        # class x_center y_center width height
        # ----------------------------------------------------

        if len(values) == 5:

            x_center, y_center, box_w, box_h = coords

            x_center *= width
            y_center *= height
            box_w *= width
            box_h *= height

            x1 = x_center - box_w / 2
            y1 = y_center - box_h / 2

            rectangle = Rectangle(
                (x1, y1),
                box_w,
                box_h,
                fill=False,
                linewidth=2
            )

            ax.add_patch(rectangle)

            ax.text(
                x1,
                max(0, y1 - 4),
                class_names[class_id],
                fontsize=8,
                bbox=dict(
                    facecolor="white",
                    alpha=0.7,
                    edgecolor="none"
                )
            )

        # ----------------------------------------------------
        # YOLO segmentation format
        #
        # class x1 y1 x2 y2 ... xn yn
        # ----------------------------------------------------

        elif len(values) > 5:

            points = []

            for i in range(0, len(coords), 2):

                if i + 1 >= len(coords):
                    break

                x = coords[i] * width
                y = coords[i + 1] * height

                points.append([x, y])

            if len(points) >= 3:

                polygon = Polygon(
                    points,
                    closed=True,
                    fill=False,
                    linewidth=2
                )

                ax.add_patch(polygon)

                label_x = points[0][0]
                label_y = points[0][1]

                ax.text(
                    label_x,
                    label_y,
                    class_names[class_id],
                    fontsize=8,
                    bbox=dict(
                        facecolor="white",
                        alpha=0.7,
                        edgecolor="none"
                    )
                )


# ============================================================
# 4. DISPLAY ALL CLASSES
# ============================================================

print("Representative examples:")
print("-" * 80)

for class_id in range(NUM_CLASSES):

    examples = selected_examples[class_id]

    print(
        f"{class_id:2d} | "
        f"{class_names[class_id]:32s} | "
        f"{len(class_examples[class_id])} images available"
    )

print()
print("Displaying representative annotated images...")
print()


# Display one class at a time so images remain readable

for class_id in range(NUM_CLASSES):

    examples = selected_examples[class_id]

    if not examples:
        print(
            f"Class {class_id}: "
            f"{class_names[class_id]} — NO EXAMPLE FOUND"
        )
        continue

    for split, image_file, label_file in examples:

        image = cv2.imread(str(image_file))

        if image is None:
            print(
                f"Could not read image: {image_file}"
            )
            continue

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        fig, ax = plt.subplots(
            figsize=(10, 7)
        )

        ax.imshow(image)

        draw_annotations(
            ax,
            image,
            label_file,
            class_id
        )

        ax.set_title(
            f"Class {class_id}: {class_names[class_id]}\n"
            f"Split: {split} | "
            f"Image: {image_file.name}"
        )

        ax.axis("off")

        plt.tight_layout()
        plt.show()

        print(
            f"Class {class_id} — {class_names[class_id]}"
        )
        print(
            f"  Split : {split}"
        )
        print(
            f"  Image : {image_file.name}"
        )
        print(
            f"  Label : {label_file.name}"
        )
        print()


# ============================================================
# 5. FINAL INSTRUCTION
# ============================================================

print("=" * 80)
print("VISUAL VERIFICATION COMPLETE")
print("=" * 80)

print("""
Please inspect especially these classes:

1. Cell High
2. Cell Low
3. Cell Medium
4. High - Multi Cell
5. High- Hot Spot
6. High- Multiple Hot Spot
7. Low- Hot Spot
8. Low- MultipleHot Spot
9. Medium- Hot Spot
10. Medium- MultipleHot Spot
11. Mexium- Hot Spot
12. Mixed- Multiple Hot Spot
13. Module-level overheating
14. Reverse Polarity Failure
15. Shading
16. String Outage

Do NOT modify or rename any dataset files yet.
""")

In [ ]:
# ============================================================
# Stage 3 — Annotation Geometry & Taxonomy Evidence
# NO DATASET FILES ARE MODIFIED
# ============================================================

from pathlib import Path
from collections import defaultdict
import yaml
import numpy as np

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

DATASET_ROOT = Path("/content/Thermal-Anomalies-2")
DATA_YAML = DATASET_ROOT / "data.yaml"

# ------------------------------------------------------------
# Load class names
# ------------------------------------------------------------

with open(DATA_YAML, "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)

names = data["names"]

if isinstance(names, dict):
    class_names = {
        int(k): str(v)
        for k, v in names.items()
    }
else:
    class_names = {
        i: str(name)
        for i, name in enumerate(names)
    }

NUM_CLASSES = len(class_names)

# ------------------------------------------------------------
# Storage
# ------------------------------------------------------------

stats = defaultdict(lambda: {
    "detection_count": 0,
    "polygon_count": 0,
    "total_count": 0,

    "box_areas": [],
    "polygon_areas": [],

    "coordinate_counts": [],

    "splits": set(),
    "images": set()
})


# ============================================================
# Helper: polygon area using shoelace formula
# ============================================================

def polygon_area(points):

    if len(points) < 3:
        return 0.0

    x = np.array([p[0] for p in points])
    y = np.array([p[1] for p in points])

    return 0.5 * abs(
        np.dot(x, np.roll(y, 1))
        -
        np.dot(y, np.roll(x, 1))
    )


# ============================================================
# Scan annotations
# ============================================================

for split in ["train", "valid", "test"]:

    label_dir = DATASET_ROOT / split / "labels"

    if not label_dir.exists():
        continue

    for label_file in label_dir.glob("*.txt"):

        text = label_file.read_text(
            encoding="utf-8",
            errors="ignore"
        ).strip()

        if not text:
            continue

        for line in text.splitlines():

            values = line.strip().split()

            if not values:
                continue

            try:
                class_id = int(float(values[0]))
            except ValueError:
                continue

            if class_id not in class_names:
                continue

            coords = list(map(float, values[1:]))

            stats[class_id]["total_count"] += 1
            stats[class_id]["splits"].add(split)
            stats[class_id]["images"].add(label_file.stem)

            # ------------------------------------------------
            # Detection annotation
            # ------------------------------------------------

            if len(values) == 5:

                x_center, y_center, width, height = coords

                area = width * height

                stats[class_id]["detection_count"] += 1
                stats[class_id]["box_areas"].append(area)

            # ------------------------------------------------
            # Segmentation annotation
            # ------------------------------------------------

            elif len(values) > 5:

                points = []

                for i in range(0, len(coords), 2):

                    if i + 1 >= len(coords):
                        break

                    points.append(
                        (coords[i], coords[i + 1])
                    )

                area = polygon_area(points)

                stats[class_id]["polygon_count"] += 1
                stats[class_id]["polygon_areas"].append(area)
                stats[class_id]["coordinate_counts"].append(
                    len(points)
                )


# ============================================================
# Print detailed analysis
# ============================================================

print("=" * 110)
print("STAGE 3 — ANNOTATION GEOMETRY & TAXONOMY EVIDENCE")
print("=" * 110)

header = (
    f"{'ID':<4}"
    f"{'Class':<32}"
    f"{'Total':>8}"
    f"{'Box':>8}"
    f"{'Poly':>8}"
    f"{'Avg Box Area':>15}"
    f"{'Avg Poly Area':>15}"
    f"{'Avg Poly Pts':>15}"
)

print(header)
print("-" * 110)

for class_id in range(NUM_CLASSES):

    s = stats[class_id]

    avg_box_area = (
        np.mean(s["box_areas"])
        if s["box_areas"]
        else 0
    )

    avg_poly_area = (
        np.mean(s["polygon_areas"])
        if s["polygon_areas"]
        else 0
    )

    avg_poly_points = (
        np.mean(s["coordinate_counts"])
        if s["coordinate_counts"]
        else 0
    )

    print(
        f"{class_id:<4}"
        f"{class_names[class_id]:<32}"
        f"{s['total_count']:>8}"
        f"{s['detection_count']:>8}"
        f"{s['polygon_count']:>8}"
        f"{avg_box_area:>15.4f}"
        f"{avg_poly_area:>15.4f}"
        f"{avg_poly_points:>15.1f}"
    )


# ============================================================
# Split coverage
# ============================================================

print("\n" + "=" * 110)
print("SPLIT COVERAGE")
print("=" * 110)

print(
    f"{'ID':<4}"
    f"{'Class':<32}"
    f"{'Train':>10}"
    f"{'Valid':>10}"
    f"{'Test':>10}"
)

print("-" * 70)

for class_id in range(NUM_CLASSES):

    split_counts = {}

    for split in ["train", "valid", "test"]:

        count = 0

        label_dir = DATASET_ROOT / split / "labels"

        if label_dir.exists():

            for label_file in label_dir.glob("*.txt"):

                text = label_file.read_text(
                    encoding="utf-8",
                    errors="ignore"
                ).strip()

                if not text:
                    continue

                for line in text.splitlines():

                    values = line.strip().split()

                    if not values:
                        continue

                    try:
                        cid = int(float(values[0]))
                    except ValueError:
                        continue

                    if cid == class_id:
                        count += 1

        split_counts[split] = count

    print(
        f"{class_id:<4}"
        f"{class_names[class_id]:<32}"
        f"{split_counts['train']:>10}"
        f"{split_counts['valid']:>10}"
        f"{split_counts['test']:>10}"
    )


# ============================================================
# Taxonomy groups
# ============================================================

print("\n" + "=" * 110)
print("CURRENT TAXONOMY PROPOSAL — EVIDENCE ONLY")
print("=" * 110)

hotspot_classes = [
    5, 6, 7, 8, 9, 10, 11, 12
]

other_classes = [
    3, 13, 14, 15, 16
]

ambiguous_classes = [
    0, 1, 2, 4
]

print("\nHOTSPOT FAMILY:")
for cid in hotspot_classes:
    print(
        f"  {cid}: {class_names[cid]} "
        f"({stats[cid]['total_count']} annotations)"
    )

print("\nOTHER FAILURE TYPES:")
for cid in other_classes:
    print(
        f"  {cid}: {class_names[cid]} "
        f"({stats[cid]['total_count']} annotations)"
    )

print("\nAMBIGUOUS / LOW-CONFIDENCE GROUP:")
for cid in ambiguous_classes:
    print(
        f"  {cid}: {class_names[cid]} "
        f"({stats[cid]['total_count']} annotations)"
    )


# ============================================================
# Hotspot combined count
# ============================================================

hotspot_total = sum(
    stats[cid]["total_count"]
    for cid in hotspot_classes
)

print("\n" + "=" * 110)
print("HOTSPOT FAMILY TOTAL")
print("=" * 110)

print(
    f"Hotspot-related annotations: {hotspot_total}"
)

print(
    f"Percentage of all annotations: "
    f"{100 * hotspot_total / sum(s['total_count'] for s in stats.values()):.2f}%"
)


# ============================================================
# Important warning
# ============================================================

print("\n" + "=" * 110)
print("IMPORTANT")
print("=" * 110)

print("""
This analysis is descriptive only.

No classes are merged.
No classes are deleted.
No annotations are converted.
No dataset files are modified.

The mixed detection/segmentation format still needs to be resolved
before YOLOv8 training.
""")

In [ ]:
# ============================================================
# Stage 3.5 — Final Taxonomy Visual Verification
# Proposed 5-class taxonomy
#
# NO DATASET FILES ARE MODIFIED
# ============================================================

from pathlib import Path
from collections import defaultdict
import random

import yaml
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

DATASET_ROOT = Path("/content/Thermal-Anomalies-2")
DATA_YAML = DATASET_ROOT / "data.yaml"

RANDOM_SEED = 42
EXAMPLES_PER_CLASS = 4

random.seed(RANDOM_SEED)

# ------------------------------------------------------------
# Original → proposed taxonomy
# ------------------------------------------------------------

TAXONOMY = {
    "Hotspot": [
        5,   # High- Hot Spot
        6,   # High- Multiple Hot Spot
        7,   # Low- Hot Spot
        8,   # Low- MultipleHot Spot
        9,   # Medium- Hot Spot
        10,  # Medium- MultipleHot Spot
        11,  # Mexium- Hot Spot
        12,  # Mixed- Multiple Hot Spot
    ],

    "Diode Failure": [
        3
    ],

    "Reverse Polarity Failure": [
        14
    ],

    "String Outage": [
        16
    ],

    "Shading": [
        15
    ],
}

# ------------------------------------------------------------
# Load original class names
# ------------------------------------------------------------

with open(DATA_YAML, "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)

names = data["names"]

if isinstance(names, dict):
    class_names = {
        int(k): str(v)
        for k, v in names.items()
    }
else:
    class_names = {
        i: str(name)
        for i, name in enumerate(names)
    }

print("=" * 90)
print("STAGE 3.5 — FINAL TAXONOMY VISUAL VERIFICATION")
print("=" * 90)

print("\nProposed taxonomy:\n")

for new_class, original_ids in TAXONOMY.items():

    print(f"{new_class}")
    print("-" * len(new_class))

    for cid in original_ids:
        print(
            f"  {cid}: {class_names[cid]}"
        )

    print()


# ============================================================
# Collect candidate images
# ============================================================

taxonomy_examples = defaultdict(list)

for split in ["train", "valid", "test"]:

    image_dir = DATASET_ROOT / split / "images"
    label_dir = DATASET_ROOT / split / "labels"

    if not image_dir.exists() or not label_dir.exists():
        continue

    image_lookup = {
        p.stem: p
        for p in image_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in {
            ".jpg", ".jpeg", ".png", ".bmp",
            ".tif", ".tiff", ".webp"
        }
    }

    for label_file in label_dir.glob("*.txt"):

        image_file = image_lookup.get(label_file.stem)

        if image_file is None:
            continue

        text = label_file.read_text(
            encoding="utf-8",
            errors="ignore"
        ).strip()

        if not text:
            continue

        class_ids = set()

        for line in text.splitlines():

            values = line.strip().split()

            if not values:
                continue

            try:
                cid = int(float(values[0]))
            except ValueError:
                continue

            class_ids.add(cid)

        for new_class, original_ids in TAXONOMY.items():

            if class_ids.intersection(original_ids):

                taxonomy_examples[new_class].append(
                    (
                        split,
                        image_file,
                        label_file
                    )
                )


# ============================================================
# Annotation drawing helper
# ============================================================

def draw_target_annotations(
    ax,
    image,
    label_file,
    target_original_ids
):

    height, width = image.shape[:2]

    text = label_file.read_text(
        encoding="utf-8",
        errors="ignore"
    ).strip()

    for line in text.splitlines():

        values = line.strip().split()

        if not values:
            continue

        try:
            class_id = int(float(values[0]))
        except ValueError:
            continue

        if class_id not in target_original_ids:
            continue

        coords = list(map(float, values[1:]))

        # ----------------------------------------------------
        # Detection box
        # ----------------------------------------------------

        if len(values) == 5:

            xc, yc, w, h = coords

            xc *= width
            yc *= height
            w *= width
            h *= height

            x1 = xc - w / 2
            y1 = yc - h / 2

            rect = Rectangle(
                (x1, y1),
                w,
                h,
                fill=False,
                linewidth=2
            )

            ax.add_patch(rect)

        # ----------------------------------------------------
        # Segmentation polygon
        # ----------------------------------------------------

        elif len(values) > 5:

            points = []

            for i in range(0, len(coords) - 1, 2):

                x = coords[i] * width
                y = coords[i + 1] * height

                points.append([x, y])

            if len(points) >= 3:

                poly = Polygon(
                    points,
                    closed=True,
                    fill=False,
                    linewidth=2
                )

                ax.add_patch(poly)


# ============================================================
# Display examples
# ============================================================

for new_class, original_ids in TAXONOMY.items():

    examples = taxonomy_examples[new_class].copy()

    random.shuffle(examples)

    examples = examples[:EXAMPLES_PER_CLASS]

    print(
        f"{new_class}: "
        f"{len(taxonomy_examples[new_class])} candidate images"
    )

    if not examples:
        print("  WARNING: no examples found")
        continue

    fig, axes = plt.subplots(
        1,
        len(examples),
        figsize=(18, 5)
    )

    if len(examples) == 1:
        axes = [axes]

    for ax, (split, image_file, label_file) in zip(
        axes,
        examples
    ):

        image = cv2.imread(str(image_file))

        if image is None:
            ax.axis("off")
            continue

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        ax.imshow(image)

        draw_target_annotations(
            ax,
            image,
            label_file,
            original_ids
        )

        ax.set_title(
            f"{split}\n{image_file.name[:22]}"
        )

        ax.axis("off")

    fig.suptitle(
        f"Proposed Class: {new_class}",
        fontsize=16
    )

    plt.tight_layout()
    plt.show()


print("\n" + "=" * 90)
print("VISUAL TAXONOMY CHECK COMPLETE")
print("=" * 90)

print("""
No files were modified.

If these five categories visually make sense,
the next step will be:

Stage 4 — Create a clean YOLOv8 segmentation dataset
with the finalized taxonomy.
""")

In [ ]:
# ============================================================
# Stage 4 — Create Clean YOLOv8 Segmentation Dataset
#
# Original dataset is NEVER modified.
#
# Output:
# /content/Solar_Thermal_Anomaly_Segmentation
# ============================================================

from pathlib import Path
from collections import defaultdict
import shutil
import yaml

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

SOURCE_ROOT = Path("/content/Thermal-Anomalies-2")

OUTPUT_ROOT = Path(
    "/content/Solar_Thermal_Anomaly_Segmentation"
)

assert SOURCE_ROOT.exists(), (
    f"Source dataset not found: {SOURCE_ROOT}"
)

# ------------------------------------------------------------
# Final taxonomy
# ------------------------------------------------------------

NEW_CLASSES = [
    "Hotspot",
    "Diode Failure",
    "Reverse Polarity Failure",
    "String Outage",
    "Shading"
]

NEW_CLASS_IDS = {
    name: i
    for i, name in enumerate(NEW_CLASSES)
}

# Original class IDs → new class ID
#
# Original:
# 5,6,7,8,9,10,11,12 = Hotspot
# 3 = Diode Failure
# 14 = Reverse Polarity Failure
# 16 = String Outage
# 15 = Shading

CLASS_MAPPING = {

    # Hotspot family
    5: 0,
    6: 0,
    7: 0,
    8: 0,
    9: 0,
    10: 0,
    11: 0,
    12: 0,

    # Other fault types
    3: 1,
    14: 2,
    16: 3,
    15: 4,
}

# ------------------------------------------------------------
# Recreate output directory safely
# ------------------------------------------------------------

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("=" * 90)
print("STAGE 4 — CREATING CLEAN YOLOv8 SEGMENTATION DATASET")
print("=" * 90)

print(f"\nSource:")
print(f"  {SOURCE_ROOT}")

print(f"\nOutput:")
print(f"  {OUTPUT_ROOT}")

print("\nFinal classes:")

for i, name in enumerate(NEW_CLASSES):
    print(f"  {i}: {name}")


# ============================================================
# Statistics
# ============================================================

statistics = {

    "train": defaultdict(int),
    "valid": defaultdict(int),
    "test": defaultdict(int)
}

source_detection_rows = 0
source_polygon_rows = 0

converted_box_rows = 0
converted_polygon_rows = 0

excluded_rows = 0

output_annotation_rows = 0

output_images = {
    "train": 0,
    "valid": 0,
    "test": 0
}


# ============================================================
# Process each split
# ============================================================

for split in ["train", "valid", "test"]:

    source_image_dir = SOURCE_ROOT / split / "images"
    source_label_dir = SOURCE_ROOT / split / "labels"

    output_image_dir = OUTPUT_ROOT / split / "images"
    output_label_dir = OUTPUT_ROOT / split / "labels"

    output_image_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    output_label_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    print(f"\nProcessing split: {split}")

    # --------------------------------------------------------
    # Copy images
    # --------------------------------------------------------

    image_files = [
        p
        for p in source_image_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in {
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp",
            ".tif",
            ".tiff",
            ".webp"
        }
    ]

    for image_file in image_files:

        destination = (
            output_image_dir / image_file.name
        )

        shutil.copy2(
            image_file,
            destination
        )

        output_images[split] += 1

    # --------------------------------------------------------
    # Convert labels
    # --------------------------------------------------------

    for source_label in sorted(
        source_label_dir.glob("*.txt")
    ):

        source_text = source_label.read_text(
            encoding="utf-8",
            errors="ignore"
        ).strip()

        if not source_text:
            continue

        output_lines = []

        for line_number, line in enumerate(
            source_text.splitlines(),
            start=1
        ):

            values = line.strip().split()

            if not values:
                continue

            try:
                original_class = int(
                    float(values[0])
                )
            except ValueError:

                print(
                    f"WARNING: invalid class ID "
                    f"in {source_label.name}, "
                    f"line {line_number}"
                )

                continue

            # ------------------------------------------------
            # Excluded class
            # ------------------------------------------------

            if original_class not in CLASS_MAPPING:

                excluded_rows += 1
                continue

            new_class = CLASS_MAPPING[
                original_class
            ]

            coords = list(
                map(
                    float,
                    values[1:]
                )
            )

            # ------------------------------------------------
            # Existing YOLO detection box
            #
            # class xc yc width height
            #
            # Convert rectangle → segmentation polygon
            # ------------------------------------------------

            if len(values) == 5:

                source_detection_rows += 1
                converted_box_rows += 1

                xc, yc, width, height = coords

                x1 = xc - width / 2
                y1 = yc - height / 2

                x2 = xc + width / 2
                y2 = yc + height / 2

                # Clamp to valid normalized range
                x1 = max(0.0, min(1.0, x1))
                y1 = max(0.0, min(1.0, y1))
                x2 = max(0.0, min(1.0, x2))
                y2 = max(0.0, min(1.0, y2))

                polygon = [
                    x1, y1,
                    x2, y1,
                    x2, y2,
                    x1, y2
                ]

                output_line = (
                    str(new_class)
                    + " "
                    + " ".join(
                        f"{value:.6f}"
                        for value in polygon
                    )
                )

                output_lines.append(
                    output_line
                )

                statistics[split][new_class] += 1
                output_annotation_rows += 1

            # ------------------------------------------------
            # Existing YOLO segmentation polygon
            #
            # class x1 y1 x2 y2 ... xn yn
            # ------------------------------------------------

            elif len(values) > 5:

                source_polygon_rows += 1
                converted_polygon_rows += 1

                # Keep polygon coordinates
                polygon = coords.copy()

                # Ensure even number of coordinates
                if len(polygon) % 2 != 0:

                    print(
                        f"WARNING: odd coordinate count "
                        f"in {source_label.name}, "
                        f"line {line_number}"
                    )

                    continue

                # Clamp coordinates
                polygon = [
                    max(
                        0.0,
                        min(1.0, value)
                    )
                    for value in polygon
                ]

                output_line = (
                    str(new_class)
                    + " "
                    + " ".join(
                        f"{value:.6f}"
                        for value in polygon
                    )
                )

                output_lines.append(
                    output_line
                )

                statistics[split][new_class] += 1
                output_annotation_rows += 1

        # ----------------------------------------------------
        # Write converted label only if it contains
        # at least one selected annotation
        # ----------------------------------------------------

        if output_lines:

            destination_label = (
                output_label_dir /
                source_label.name
            )

            destination_label.write_text(
                "\n".join(output_lines),
                encoding="utf-8"
            )


# ============================================================
# Create data.yaml
# ============================================================

data_yaml = {
    "path": str(OUTPUT_ROOT),

    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",

    "names": {
        i: name
        for i, name in enumerate(NEW_CLASSES)
    }
}

with open(
    OUTPUT_ROOT / "data.yaml",
    "w",
    encoding="utf-8"
) as f:

    yaml.safe_dump(
        data_yaml,
        f,
        sort_keys=False
    )


# ============================================================
# Print conversion summary
# ============================================================

print("\n" + "=" * 90)
print("CONVERSION SUMMARY")
print("=" * 90)

print(
    f"Source detection rows      : "
    f"{source_detection_rows}"
)

print(
    f"Source polygon rows        : "
    f"{source_polygon_rows}"
)

print(
    f"Converted box → polygon    : "
    f"{converted_box_rows}"
)

print(
    f"Existing polygons retained : "
    f"{converted_polygon_rows}"
)

print(
    f"Excluded annotations       : "
    f"{excluded_rows}"
)

print(
    f"Output annotations         : "
    f"{output_annotation_rows}"
)

print("\nOutput images:")

for split in ["train", "valid", "test"]:
    print(
        f"  {split:<6}: "
        f"{output_images[split]}"
    )


# ============================================================
# Class distribution
# ============================================================

print("\n" + "=" * 90)
print("OUTPUT CLASS DISTRIBUTION")
print("=" * 90)

print(
    f"{'ID':<5}"
    f"{'Class':<30}"
    f"{'Train':>12}"
    f"{'Valid':>12}"
    f"{'Test':>12}"
    f"{'Total':>12}"
)

print("-" * 85)

for class_id, class_name in enumerate(
    NEW_CLASSES
):

    train_count = statistics["train"][class_id]
    valid_count = statistics["valid"][class_id]
    test_count = statistics["test"][class_id]

    total = (
        train_count
        + valid_count
        + test_count
    )

    print(
        f"{class_id:<5}"
        f"{class_name:<30}"
        f"{train_count:>12}"
        f"{valid_count:>12}"
        f"{test_count:>12}"
        f"{total:>12}"
    )


# ============================================================
# Final message
# ============================================================

print("\n" + "=" * 90)
print("STAGE 4 DATASET CREATION COMPLETE")
print("=" * 90)

print(f"""
Clean segmentation dataset created at:

{OUTPUT_ROOT}

Original dataset remains untouched:

{SOURCE_ROOT}

The next step is NOT training yet.

We will first validate:
1. data.yaml
2. class IDs
3. segmentation coordinate validity
4. image/label consistency
5. empty labels
6. final class distribution

Only after those checks pass will we install/setup YOLOv8 and train.
""")

In [ ]:
# ============================================================
# Stage 4.1 — Final Clean Dataset Validation
#
# NO FILES ARE MODIFIED
# ============================================================

from pathlib import Path
from collections import defaultdict
import yaml
import cv2

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

DATASET_ROOT = Path(
    "/content/Solar_Thermal_Anomaly_Segmentation"
)

DATA_YAML = DATASET_ROOT / "data.yaml"

assert DATASET_ROOT.exists(), (
    f"Dataset not found: {DATASET_ROOT}"
)

assert DATA_YAML.exists(), (
    f"data.yaml not found: {DATA_YAML}"
)

# ------------------------------------------------------------
# Expected taxonomy
# ------------------------------------------------------------

EXPECTED_CLASSES = {
    0: "Hotspot",
    1: "Diode Failure",
    2: "Reverse Polarity Failure",
    3: "String Outage",
    4: "Shading"
}

NUM_CLASSES = len(EXPECTED_CLASSES)

print("=" * 90)
print("STAGE 4.1 — FINAL CLEAN DATASET VALIDATION")
print("=" * 90)

print(f"\nDataset:")
print(DATASET_ROOT)


# ============================================================
# 1. Validate data.yaml
# ============================================================

print("\n" + "=" * 90)
print("1. data.yaml VALIDATION")
print("=" * 90)

with open(
    DATA_YAML,
    "r",
    encoding="utf-8"
) as f:

    data = yaml.safe_load(f)

yaml_errors = []

# Check names
names = data.get("names")

if isinstance(names, dict):

    yaml_names = {
        int(k): str(v)
        for k, v in names.items()
    }

elif isinstance(names, list):

    yaml_names = {
        i: str(v)
        for i, v in enumerate(names)
    }

else:

    yaml_names = {}
    yaml_errors.append(
        "Invalid or missing 'names' field"
    )

print("Classes in data.yaml:")

for class_id in sorted(yaml_names):

    print(
        f"  {class_id}: "
        f"{yaml_names[class_id]}"
    )

if yaml_names != EXPECTED_CLASSES:

    yaml_errors.append(
        "Class definitions do not match expected taxonomy"
    )

# Check paths
for key in ["train", "val", "test"]:

    if key not in data:

        yaml_errors.append(
            f"Missing '{key}' path"
        )

# ------------------------------------------------------------
# Result
# ------------------------------------------------------------

if yaml_errors:

    print("\n⚠ data.yaml problems:")

    for error in yaml_errors:
        print("  -", error)

else:

    print("\n✓ data.yaml is valid")
    print("✓ Five-class taxonomy matches")


# ============================================================
# 2. Storage for validation
# ============================================================

total_images = 0
total_labels = 0
total_annotations = 0

empty_label_files = []
missing_label_files = []
missing_image_files = []
corrupt_images = []

invalid_class_ids = []
non_numeric_values = []

odd_coordinate_counts = []
too_few_polygon_points = []

coordinates_outside_range = []

zero_area_polygons = []

duplicate_annotations = []

class_counts = defaultdict(int)

split_counts = {
    "train": 0,
    "valid": 0,
    "test": 0
}

split_annotation_counts = {
    "train": 0,
    "valid": 0,
    "test": 0
}


# ============================================================
# 3. Validate every split
# ============================================================

for split in ["train", "valid", "test"]:

    print("\n" + "=" * 90)
    print(f"2. VALIDATING SPLIT: {split.upper()}")
    print("=" * 90)

    image_dir = DATASET_ROOT / split / "images"
    label_dir = DATASET_ROOT / split / "labels"

    assert image_dir.exists(), (
        f"Missing image directory: {image_dir}"
    )

    assert label_dir.exists(), (
        f"Missing label directory: {label_dir}"
    )

    image_files = {
        p.stem: p
        for p in image_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in {
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp",
            ".tif",
            ".tiff",
            ".webp"
        }
    }

    label_files = {
        p.stem: p
        for p in label_dir.glob("*.txt")
    }

    split_counts[split] = len(image_files)

    total_images += len(image_files)
    total_labels += len(label_files)

    print(
        f"Images : {len(image_files)}"
    )

    print(
        f"Labels : {len(label_files)}"
    )

    # --------------------------------------------------------
    # Image → label consistency
    # --------------------------------------------------------

    for stem in image_files:

        if stem not in label_files:

            missing_label_files.append(
                (split, stem)
            )

    # --------------------------------------------------------
    # Label → image consistency
    # --------------------------------------------------------

    for stem in label_files:

        if stem not in image_files:

            missing_image_files.append(
                (split, stem)
            )

    # --------------------------------------------------------
    # Image integrity
    # --------------------------------------------------------

    for stem, image_path in image_files.items():

        image = cv2.imread(
            str(image_path)
        )

        if image is None:

            corrupt_images.append(
                (split, image_path.name)
            )

    # --------------------------------------------------------
    # Annotation validation
    # --------------------------------------------------------

    for stem, label_path in label_files.items():

        text = label_path.read_text(
            encoding="utf-8",
            errors="ignore"
        ).strip()

        if not text:

            empty_label_files.append(
                (split, label_path.name)
            )

            continue

        seen_annotations = set()

        for line_number, line in enumerate(
            text.splitlines(),
            start=1
        ):

            values = line.strip().split()

            if not values:
                continue

            total_annotations += 1
            split_annotation_counts[split] += 1

            # ------------------------------------------------
            # Class ID
            # ------------------------------------------------

            try:

                class_value = float(
                    values[0]
                )

                if not class_value.is_integer():

                    invalid_class_ids.append(
                        (
                            split,
                            label_path.name,
                            line_number,
                            values[0]
                        )
                    )

                    continue

                class_id = int(
                    class_value
                )

            except ValueError:

                invalid_class_ids.append(
                    (
                        split,
                        label_path.name,
                        line_number,
                        values[0]
                    )
                )

                continue

            if class_id not in EXPECTED_CLASSES:

                invalid_class_ids.append(
                    (
                        split,
                        label_path.name,
                        line_number,
                        class_id
                    )
                )

                continue

            class_counts[class_id] += 1

            # ------------------------------------------------
            # Segmentation format
            #
            # class x1 y1 x2 y2 ... xn yn
            # ------------------------------------------------

            if len(values) < 7:

                too_few_polygon_points.append(
                    (
                        split,
                        label_path.name,
                        line_number,
                        len(values)
                    )
                )

                continue

            coordinate_values = []

            try:

                coordinate_values = [
                    float(v)
                    for v in values[1:]
                ]

            except ValueError:

                non_numeric_values.append(
                    (
                        split,
                        label_path.name,
                        line_number
                    )
                )

                continue

            # ------------------------------------------------
            # Even coordinate count
            # ------------------------------------------------

            if len(coordinate_values) % 2 != 0:

                odd_coordinate_counts.append(
                    (
                        split,
                        label_path.name,
                        line_number,
                        len(coordinate_values)
                    )
                )

                continue

            # ------------------------------------------------
            # Coordinate range
            # ------------------------------------------------

            outside = [
                value
                for value in coordinate_values
                if value < 0.0 or value > 1.0
            ]

            if outside:

                coordinates_outside_range.append(
                    (
                        split,
                        label_path.name,
                        line_number,
                        min(coordinate_values),
                        max(coordinate_values)
                    )
                )

            # ------------------------------------------------
            # Polygon area
            # ------------------------------------------------

            points = []

            for i in range(
                0,
                len(coordinate_values),
                2
            ):

                x = coordinate_values[i]
                y = coordinate_values[i + 1]

                points.append(
                    (x, y)
                )

            # Shoelace formula
            area = 0.0

            if len(points) >= 3:

                x_values = [
                    p[0]
                    for p in points
                ]

                y_values = [
                    p[1]
                    for p in points
                ]

                for i in range(
                    len(points)
                ):

                    j = (
                        i + 1
                    ) % len(points)

                    area += (
                        x_values[i]
                        * y_values[j]
                        -
                        x_values[j]
                        * y_values[i]
                    )

                area = abs(area) / 2.0

            if area <= 0:

                zero_area_polygons.append(
                    (
                        split,
                        label_path.name,
                        line_number,
                        area
                    )
                )

            # ------------------------------------------------
            # Duplicate annotation
            # ------------------------------------------------

            normalized_annotation = (
                class_id,
                tuple(
                    round(
                        value,
                        6
                    )
                    for value in coordinate_values
                )
            )

            if normalized_annotation in seen_annotations:

                duplicate_annotations.append(
                    (
                        split,
                        label_path.name,
                        line_number
                    )
                )

            else:

                seen_annotations.add(
                    normalized_annotation
                )


# ============================================================
# 4. Print validation summary
# ============================================================

print("\n" + "=" * 90)
print("3. DATASET VALIDATION SUMMARY")
print("=" * 90)

print(
    f"Total images              : {total_images}"
)

print(
    f"Total label files         : {total_labels}"
)

print(
    f"Total annotations         : {total_annotations}"
)

print(
    f"Empty label files         : {len(empty_label_files)}"
)

print(
    f"Missing label files       : {len(missing_label_files)}"
)

print(
    f"Missing image files       : {len(missing_image_files)}"
)

print(
    f"Corrupt images            : {len(corrupt_images)}"
)

print(
    f"Invalid class IDs         : {len(invalid_class_ids)}"
)

print(
    f"Non-numeric coordinates   : {len(non_numeric_values)}"
)

print(
    f"Odd coordinate counts     : {len(odd_coordinate_counts)}"
)

print(
    f"Too few polygon points    : {len(too_few_polygon_points)}"
)

print(
    f"Out-of-range coordinates  : {len(coordinates_outside_range)}"
)

print(
    f"Zero-area polygons        : {len(zero_area_polygons)}"
)

print(
    f"Duplicate annotations     : {len(duplicate_annotations)}"
)


# ============================================================
# 5. Split summary
# ============================================================

print("\n" + "=" * 90)
print("4. SPLIT SUMMARY")
print("=" * 90)

print(
    f"{'Split':<10}"
    f"{'Images':>12}"
    f"{'Annotations':>16}"
)

print("-" * 40)

for split in ["train", "valid", "test"]:

    print(
        f"{split:<10}"
        f"{split_counts[split]:>12}"
        f"{split_annotation_counts[split]:>16}"
    )

print("-" * 40)

print(
    f"{'TOTAL':<10}"
    f"{total_images:>12}"
    f"{total_annotations:>16}"
)


# ============================================================
# 6. Class distribution
# ============================================================

print("\n" + "=" * 90)
print("5. FINAL CLASS DISTRIBUTION")
print("=" * 90)

print(
    f"{'ID':<5}"
    f"{'Class':<30}"
    f"{'Annotations':>15}"
)

print("-" * 55)

for class_id in range(NUM_CLASSES):

    print(
        f"{class_id:<5}"
        f"{EXPECTED_CLASSES[class_id]:<30}"
        f"{class_counts[class_id]:>15}"
    )


# ============================================================
# 7. Pass / fail
# ============================================================

print("\n" + "=" * 90)
print("6. FINAL VALIDATION RESULT")
print("=" * 90)

errors = (
    yaml_errors
    + invalid_class_ids
    + non_numeric_values
    + odd_coordinate_counts
    + too_few_polygon_points
    + coordinates_outside_range
    + zero_area_polygons
    + duplicate_annotations
    + missing_label_files
    + missing_image_files
    + corrupt_images
)

if errors:

    print("⚠ DATASET VALIDATION FOUND ISSUES")
    print()
    print(
        "Do NOT train yet."
    )

else:

    print("✓ DATASET VALIDATION PASSED")
    print()
    print(
        "The cleaned dataset is structurally valid "
        "for YOLOv8 segmentation."
    )

print("\n" + "=" * 90)
print("NO FILES WERE MODIFIED")
print("=" * 90)

In [ ]:
# ============================================================
# Stage 4.2 — Investigate Missing Labels
#
# Compare cleaned dataset against ORIGINAL dataset.
# NO FILES ARE MODIFIED.
# ============================================================

from pathlib import Path
from collections import defaultdict
import yaml

SOURCE_ROOT = Path(
    "/content/Thermal-Anomalies-2"
)

CLEAN_ROOT = Path(
    "/content/Solar_Thermal_Anomaly_Segmentation"
)

DATA_YAML = SOURCE_ROOT / "data.yaml"

# ------------------------------------------------------------
# Load original class names
# ------------------------------------------------------------

with open(
    DATA_YAML,
    "r",
    encoding="utf-8"
) as f:
    data = yaml.safe_load(f)

names = data["names"]

if isinstance(names, dict):
    original_names = {
        int(k): str(v)
        for k, v in names.items()
    }
else:
    original_names = {
        i: str(v)
        for i, v in enumerate(names)
    }

# ------------------------------------------------------------
# Original → final taxonomy mapping
# ------------------------------------------------------------

CLASS_MAPPING = {
    5: 0,
    6: 0,
    7: 0,
    8: 0,
    9: 0,
    10: 0,
    11: 0,
    12: 0,

    3: 1,
    14: 2,
    16: 3,
    15: 4,
}

FINAL_NAMES = {
    0: "Hotspot",
    1: "Diode Failure",
    2: "Reverse Polarity Failure",
    3: "String Outage",
    4: "Shading"
}

# ------------------------------------------------------------
# Find missing-label images
# ------------------------------------------------------------

missing_by_split = defaultdict(list)

for split in ["train", "valid", "test"]:

    clean_image_dir = (
        CLEAN_ROOT / split / "images"
    )

    clean_label_dir = (
        CLEAN_ROOT / split / "labels"
    )

    image_files = [
        p for p in clean_image_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in {
            ".jpg", ".jpeg", ".png", ".bmp",
            ".tif", ".tiff", ".webp"
        }
    ]

    label_stems = {
        p.stem
        for p in clean_label_dir.glob("*.txt")
    }

    for image_file in image_files:

        if image_file.stem not in label_stems:

            missing_by_split[split].append(
                image_file.stem
            )


# ============================================================
# Analyze original annotations
# ============================================================

print("=" * 100)
print("STAGE 4.2 — MISSING LABEL INVESTIGATION")
print("=" * 100)

total_missing = 0

category_counts = defaultdict(int)

examples = defaultdict(list)

for split in ["train", "valid", "test"]:

    original_label_dir = (
        SOURCE_ROOT / split / "labels"
    )

    print(
        f"\n{split.upper()}"
    )
    print("-" * 100)

    for stem in missing_by_split[split]:

        total_missing += 1

        original_label = (
            original_label_dir /
            f"{stem}.txt"
        )

        # ----------------------------------------------------
        # Original label missing completely
        # ----------------------------------------------------

        if not original_label.exists():

            category = "ORIGINAL_LABEL_MISSING"

            category_counts[category] += 1

            if len(examples[category]) < 10:
                examples[category].append(
                    (split, stem)
                )

            continue

        text = original_label.read_text(
            encoding="utf-8",
            errors="ignore"
        ).strip()

        # ----------------------------------------------------
        # Original label empty
        # ----------------------------------------------------

        if not text:

            category = "ORIGINAL_LABEL_EMPTY"

            category_counts[category] += 1

            if len(examples[category]) < 10:
                examples[category].append(
                    (split, stem)
                )

            continue

        # ----------------------------------------------------
        # Examine original class IDs
        # ----------------------------------------------------

        original_classes = []
        final_classes = []

        for line in text.splitlines():

            values = line.strip().split()

            if not values:
                continue

            try:
                cid = int(float(values[0]))
            except ValueError:
                continue

            original_classes.append(cid)

            if cid in CLASS_MAPPING:
                final_classes.append(
                    CLASS_MAPPING[cid]
                )

        # ----------------------------------------------------
        # Categorize
        # ----------------------------------------------------

        unique_original = sorted(
            set(original_classes)
        )

        unique_final = sorted(
            set(final_classes)
        )

        if not original_classes:

            category = "NO_VALID_ORIGINAL_ANNOTATION"

        elif final_classes:

            category = (
                "ERROR_SELECTED_CLASS_NOT_CONVERTED"
            )

        else:

            category = (
                "ONLY_EXCLUDED_CLASSES"
            )

        category_counts[category] += 1

        if len(examples[category]) < 10:

            examples[category].append(
                (
                    split,
                    stem,
                    unique_original,
                    [
                        original_names[c]
                        for c in unique_original
                        if c in original_names
                    ]
                )
            )


# ============================================================
# Print summary
# ============================================================

print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)

print(
    f"Total missing labels investigated: "
    f"{total_missing}"
)

for category, count in sorted(
    category_counts.items()
):

    print(
        f"{category:<45} {count:>5}"
    )


# ============================================================
# Print examples
# ============================================================

print("\n" + "=" * 100)
print("EXAMPLES")
print("=" * 100)

for category in sorted(examples):

    print(
        f"\n[{category}]"
    )

    for item in examples[category]:

        if len(item) == 2:

            print(
                f"  Split={item[0]} | "
                f"Image={item[1]}"
            )

        else:

            split, stem, ids, names_list = item

            print(
                f"  Split={split} | "
                f"Image={stem}"
            )

            print(
                f"    Original IDs   : {ids}"
            )

            print(
                f"    Original names : {names_list}"
            )


# ============================================================
# Final interpretation
# ============================================================

print("\n" + "=" * 100)
print("INTERPRETATION")
print("=" * 100)

if category_counts[
    "ERROR_SELECTED_CLASS_NOT_CONVERTED"
] > 0:

    print(
        "⚠ PROBLEM: Some images contain a selected "
        "class in the original dataset but have no "
        "label in the cleaned dataset."
    )

    print(
        "Do NOT train yet."
    )

else:

    print(
        "✓ No selected-class annotations were lost."
    )

    print(
        "The missing labels are likely caused by "
        "background images or images containing only "
        "excluded taxonomy classes."
    )

print("\nNO FILES WERE MODIFIED.")

In [ ]:
# ============================================================
# Stage 4.3 — Visualize Final Clean Dataset
#
# Displays random samples with segmentation annotations.
# NO FILES ARE MODIFIED.
# ============================================================

from pathlib import Path
import random
import cv2
import matplotlib.pyplot as plt
import yaml

DATASET_ROOT = Path(
    "/content/Solar_Thermal_Anomaly_Segmentation"
)

DATA_YAML = DATASET_ROOT / "data.yaml"

# ------------------------------------------------------------
# Load class names
# ------------------------------------------------------------

with open(
    DATA_YAML,
    "r",
    encoding="utf-8"
) as f:
    data = yaml.safe_load(f)

names = data["names"]

if isinstance(names, dict):
    CLASS_NAMES = {
        int(k): str(v)
        for k, v in names.items()
    }
else:
    CLASS_NAMES = {
        i: str(v)
        for i, v in enumerate(names)
    }


# ------------------------------------------------------------
# Select images
# ------------------------------------------------------------

random.seed(42)

selected_images = []

for split in ["train", "valid", "test"]:

    image_dir = (
        DATASET_ROOT /
        split /
        "images"
    )

    images = [
        p for p in image_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in {
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp",
            ".tif",
            ".tiff",
            ".webp"
        }
    ]

    random.shuffle(images)

    for image_path in images:

        label_path = (
            DATASET_ROOT /
            split /
            "labels" /
            f"{image_path.stem}.txt"
        )

        if label_path.exists():

            text = label_path.read_text(
                encoding="utf-8",
                errors="ignore"
            ).strip()

            if text:

                selected_images.append(
                    (split, image_path, label_path)
                )

# Select 6 samples
selected_images = random.sample(
    selected_images,
    min(6, len(selected_images))
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2,
    3,
    figsize=(18, 11)
)

axes = axes.flatten()

for ax, (split, image_path, label_path) in zip(
    axes,
    selected_images
):

    image = cv2.imread(
        str(image_path)
    )

    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    height, width = image.shape[:2]

    ax.imshow(image)

    label_text = label_path.read_text(
        encoding="utf-8",
        errors="ignore"
    ).strip()

    annotation_classes = []

    for line in label_text.splitlines():

        values = line.split()

        if len(values) < 7:
            continue

        class_id = int(
            float(values[0])
        )

        coords = [
            float(v)
            for v in values[1:]
        ]

        points = []

        for i in range(
            0,
            len(coords),
            2
        ):

            x = int(
                coords[i] * width
            )

            y = int(
                coords[i + 1] * height
            )

            points.append(
                [x, y]
            )

        if len(points) >= 3:

            points = (
                __import__("numpy")
                .array(
                    points,
                    dtype="int32"
                )
            )

            cv2.polylines(
                image,
                [points],
                isClosed=True,
                color=(255, 0, 0),
                thickness=2
            )

            annotation_classes.append(
                CLASS_NAMES[class_id]
            )

    # Redraw image with polygons
    ax.clear()

    ax.imshow(image)

    ax.set_title(
        f"{split.upper()}\n"
        f"{image_path.name}\n"
        f"Classes: {', '.join(sorted(set(annotation_classes)))}",
        fontsize=9
    )

    ax.axis("off")


plt.tight_layout()
plt.show()

print(
    f"Displayed {len(selected_images)} "
    f"clean segmentation samples."
)

print("NO FILES WERE MODIFIED.")

In [ ]:
# ============================================================
# Stage 5.1 — YOLOv8 Segmentation Setup
#
# Model setup only — NO TRAINING YET
# ============================================================

%pip install -q ultralytics

from ultralytics import YOLO
import ultralytics
import torch

print("=" * 80)
print("YOLOv8 SEGMENTATION SETUP")
print("=" * 80)

print("\nUltralytics version:")
print(ultralytics.__version__)

print("\nPyTorch version:")
print(torch.__version__)

print("\nCUDA available:")
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

# ------------------------------------------------------------
# Dataset configuration
# ------------------------------------------------------------

DATASET_ROOT = (
    "/content/Solar_Thermal_Anomaly_Segmentation"
)

DATA_YAML = (
    f"{DATASET_ROOT}/data.yaml"
)

# ------------------------------------------------------------
# Load YOLOv8 segmentation model
# ------------------------------------------------------------

MODEL_NAME = "yolov8n-seg.pt"

model = YOLO(MODEL_NAME)

print("\nModel loaded successfully:")
print(MODEL_NAME)

print("\nDataset configuration:")
print(DATA_YAML)

print("\nNumber of classes:")
print(5)

print("\nClass names:")
print([
    "Hotspot",
    "Diode Failure",
    "Reverse Polarity Failure",
    "String Outage",
    "Shading"
])

print("\n" + "=" * 80)
print("STAGE 5.1 COMPLETE")
print("=" * 80)

print("✓ Ultralytics installed")
print("✓ YOLOv8 segmentation model loaded")
print("✓ Clean dataset path configured")
print("✓ Five-class taxonomy configured")
print("✓ NO TRAINING PERFORMED")

In [ ]:
# ============================================================
# Stage 5.2 — YOLOv8 Dataset Compatibility Check
#
# NO TRAINING
# NO FILES MODIFIED
# ============================================================

from ultralytics import YOLO

DATASET_ROOT = (
    "/content/Solar_Thermal_Anomaly_Segmentation"
)

DATA_YAML = (
    f"{DATASET_ROOT}/data.yaml"
)

MODEL_NAME = "yolov8n-seg.pt"

print("=" * 80)
print("STAGE 5.2 — YOLOv8 DATASET COMPATIBILITY CHECK")
print("=" * 80)

# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

model = YOLO(MODEL_NAME)

print("\n✓ Model loaded:")
print(MODEL_NAME)

# ------------------------------------------------------------
# Load dataset configuration through Ultralytics
# ------------------------------------------------------------

try:

    from ultralytics.data.utils import check_det_dataset

    dataset_info = check_det_dataset(
        DATA_YAML
    )

    print("\n✓ Ultralytics successfully loaded data.yaml")

    print("\nDataset paths:")

    print(
        "  Train:",
        dataset_info.get("train")
    )

    print(
        "  Validation:",
        dataset_info.get("val")
    )

    print(
        "  Test:",
        dataset_info.get("test")
    )

    print("\nNumber of classes:")

    print(
        "  nc:",
        dataset_info.get("nc")
    )

    print("\nClass names:")

    print(
        " ",
        dataset_info.get("names")
    )

    print("\n" + "=" * 80)

    if dataset_info.get("nc") != 5:

        print(
            "⚠ WARNING: Expected 5 classes."
        )

    else:

        print(
            "✓ Number of classes = 5"
        )

    print("=" * 80)

    print("\nSTAGE 5.2 COMPLETE")
    print("✓ Dataset configuration accepted by Ultralytics")
    print("✓ YOLOv8 segmentation model ready")
    print("✓ No training performed")
    print("✓ No files modified")

except Exception as e:

    print("\n" + "=" * 80)
    print("⚠ DATASET COMPATIBILITY CHECK FAILED")
    print("=" * 80)

    print("\nError:")
    print(e)

    print("\nDo NOT start training yet.")

In [ ]:
# ============================================================
# Stage 5.3 — YOLOv8n Segmentation Baseline Training
# ============================================================

from ultralytics import YOLO
import torch

DATASET_ROOT = (
    "/content/Solar_Thermal_Anomaly_Segmentation"
)

DATA_YAML = (
    f"{DATASET_ROOT}/data.yaml"
)

MODEL_NAME = "yolov8n-seg.pt"

print("=" * 80)
print("STAGE 5.3 — YOLOv8n SEGMENTATION BASELINE TRAINING")
print("=" * 80)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("\nGPU available:", torch.cuda.is_available())
print("Training device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

# ------------------------------------------------------------
# Load pretrained YOLOv8 segmentation model
# ------------------------------------------------------------

model = YOLO(MODEL_NAME)

# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

results = model.train(
    data=DATA_YAML,

    epochs=50,

    imgsz=640,

    batch=16,

    device=DEVICE,

    workers=2,

    pretrained=True,

    project="/content/solar_panel_runs",

    name="yolov8n_seg_baseline",

    exist_ok=True,

    patience=10,

    save=True,

    plots=True,

    verbose=True
)

print("\n" + "=" * 80)
print("BASELINE TRAINING FINISHED")
print("=" * 80)

print(
    "\nBest model should be saved at:"
)

print(
    "/content/solar_panel_runs/"
    "yolov8n_seg_baseline/weights/best.pt"
)

print("\nTraining results directory:")

print(
    "/content/solar_panel_runs/"
    "yolov8n_seg_baseline/"
)

In [ ]:
# ============================================================
# Stage 6.1 — Final Test Set Evaluation
#
# Evaluates the BEST trained model on the unseen test set.
# NO TRAINING
# ============================================================

from ultralytics import YOLO
import torch

MODEL_PATH = (
    "/content/solar_panel_runs/"
    "yolov8n_seg_baseline/weights/best.pt"
)

DATA_YAML = (
    "/content/Solar_Thermal_Anomaly_Segmentation/"
    "data.yaml"
)

print("=" * 80)
print("STAGE 6.1 — TEST SET EVALUATION")
print("=" * 80)

# ------------------------------------------------------------
# Load best trained model
# ------------------------------------------------------------

model = YOLO(MODEL_PATH)

print("\n✓ Best model loaded:")
print(MODEL_PATH)

# ------------------------------------------------------------
# Evaluate on TEST split
# ------------------------------------------------------------

test_results = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project="/content/solar_panel_runs",
    name="yolov8n_seg_test",
    exist_ok=True
)

print("\n" + "=" * 80)
print("TEST EVALUATION COMPLETE")
print("=" * 80)

print("\nTest results saved to:")

print(
    "/content/solar_panel_runs/"
    "yolov8n_seg_test/"
)

print("\nBest trained model remains at:")

print(MODEL_PATH)

In [ ]:
# ============================================================
# Stage 6.2 — Generate Predictions on Unseen Test Images
#
# Uses the trained BEST model.
# NO TRAINING
# ============================================================

from ultralytics import YOLO
from pathlib import Path
import random

MODEL_PATH = (
    "/content/solar_panel_runs/"
    "yolov8n_seg_baseline/weights/best.pt"
)

TEST_IMAGE_DIR = Path(
    "/content/Solar_Thermal_Anomaly_Segmentation/"
    "test/images"
)

OUTPUT_DIR = (
    "/content/solar_panel_runs/"
    "yolov8n_seg_test_predictions"
)

print("=" * 80)
print("STAGE 6.2 — TEST IMAGE PREDICTIONS")
print("=" * 80)

# ------------------------------------------------------------
# Load trained model
# ------------------------------------------------------------

model = YOLO(MODEL_PATH)

print("\n✓ Best model loaded")

# ------------------------------------------------------------
# Collect test images
# ------------------------------------------------------------

image_files = [
    p for p in TEST_IMAGE_DIR.iterdir()
    if p.is_file()
    and p.suffix.lower() in {
        ".jpg",
        ".jpeg",
        ".png",
        ".bmp",
        ".tif",
        ".tiff",
        ".webp"
    }
]

print(
    f"Total test images available: "
    f"{len(image_files)}"
)

# ------------------------------------------------------------
# Select reproducible sample
# ------------------------------------------------------------

random.seed(42)

selected_images = random.sample(
    image_files,
    min(12, len(image_files))
)

print(
    f"Generating predictions for "
    f"{len(selected_images)} test images..."
)

# ------------------------------------------------------------
# Run inference
# ------------------------------------------------------------

results = model.predict(
    source=[
        str(p)
        for p in selected_images
    ],

    imgsz=640,

    conf=0.25,

    device=0,

    save=True,

    project="/content/solar_panel_runs",

    name="yolov8n_seg_test_predictions",

    exist_ok=True,

    verbose=False
)

print("\n" + "=" * 80)
print("PREDICTION GENERATION COMPLETE")
print("=" * 80)

print("\nPredictions saved to:")

print(OUTPUT_DIR)

print("\nImages predicted:")

for image_path in selected_images:
    print(
        " -",
        image_path.name
    )

print("\n✓ Predictions generated successfully")
print("✓ NO MODEL TRAINING PERFORMED")

In [ ]:
# ============================================================
# Stage 6.3 — Display Test Predictions
#
# Displays the 12 generated prediction images.
# NO TRAINING
# ============================================================

from pathlib import Path
from IPython.display import display, Image

PREDICTION_DIR = Path(
    "/content/solar_panel_runs/"
    "yolov8n_seg_test_predictions"
)

prediction_images = [
    p for p in PREDICTION_DIR.iterdir()
    if p.is_file()
    and p.suffix.lower() in {
        ".jpg",
        ".jpeg",
        ".png"
    }
]

prediction_images = sorted(
    prediction_images
)

print("=" * 80)
print("STAGE 6.3 — DISPLAY TEST PREDICTIONS")
print("=" * 80)

print(
    f"\nPrediction images found: "
    f"{len(prediction_images)}"
)

for image_path in prediction_images:

    print("\n" + "-" * 80)
    print(image_path.name)

    display(
        Image(
            filename=str(image_path),
            width=700
        )
    )

print("\n" + "=" * 80)
print("STAGE 6.3 COMPLETE")
print("=" * 80)

print("✓ Prediction images displayed")
print("✓ NO FILES MODIFIED")
print("✓ NO TRAINING PERFORMED")


In [ ]:
# ============================================================
# Stage 7.1 — Test Prediction Error Analysis
#
# Compares ground-truth classes with model predictions.
# NO TRAINING
# ============================================================

from pathlib import Path
from collections import defaultdict, Counter
from ultralytics import YOLO
import yaml
import numpy as np

DATASET_ROOT = Path(
    "/content/Solar_Thermal_Anomaly_Segmentation"
)

TEST_IMAGE_DIR = (
    DATASET_ROOT / "test" / "images"
)

TEST_LABEL_DIR = (
    DATASET_ROOT / "test" / "labels"
)

MODEL_PATH = (
    "/content/solar_panel_runs/"
    "yolov8n_seg_baseline/weights/best.pt"
)

DATA_YAML = DATASET_ROOT / "data.yaml"

# ------------------------------------------------------------
# Load class names
# ------------------------------------------------------------

with open(DATA_YAML, "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)

names = data["names"]

if isinstance(names, dict):
    CLASS_NAMES = {
        int(k): str(v)
        for k, v in names.items()
    }
else:
    CLASS_NAMES = {
        i: str(v)
        for i, v in enumerate(names)
    }

# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

model = YOLO(MODEL_PATH)

print("=" * 80)
print("STAGE 7.1 — TEST PREDICTION ERROR ANALYSIS")
print("=" * 80)

# ------------------------------------------------------------
# Ground-truth class presence per image
# ------------------------------------------------------------

ground_truth = {}

test_images = sorted([
    p for p in TEST_IMAGE_DIR.iterdir()
    if p.is_file()
    and p.suffix.lower() in {
        ".jpg", ".jpeg", ".png",
        ".bmp", ".tif", ".tiff", ".webp"
    }
])

for image_path in test_images:

    label_path = (
        TEST_LABEL_DIR /
        f"{image_path.stem}.txt"
    )

    classes = []

    if label_path.exists():

        text = label_path.read_text(
            encoding="utf-8",
            errors="ignore"
        ).strip()

        for line in text.splitlines():

            values = line.split()

            if not values:
                continue

            try:
                class_id = int(float(values[0]))
                classes.append(class_id)
            except ValueError:
                pass

    ground_truth[image_path.name] = set(classes)

# ------------------------------------------------------------
# Model predictions
# ------------------------------------------------------------

print(
    f"\nRunning inference on "
    f"{len(test_images)} test images..."
)

results = model.predict(
    source=[
        str(p)
        for p in test_images
    ],
    imgsz=640,
    conf=0.25,
    device=0,
    verbose=False
)

# ------------------------------------------------------------
# Compare image-level class presence
# ------------------------------------------------------------

tp = Counter()
fp = Counter()
fn = Counter()

prediction_counts = Counter()
ground_truth_counts = Counter()

for image_path, result in zip(
    test_images,
    results
):

    true_classes = ground_truth[
        image_path.name
    ]

    predicted_classes = set()

    if result.boxes is not None:

        for cls in result.boxes.cls.tolist():

            predicted_classes.add(
                int(cls)
            )

    for class_id in true_classes:
        ground_truth_counts[class_id] += 1

    for class_id in predicted_classes:
        prediction_counts[class_id] += 1

    for class_id in range(len(CLASS_NAMES)):

        actual = class_id in true_classes
        predicted = class_id in predicted_classes

        if actual and predicted:
            tp[class_id] += 1

        elif predicted and not actual:
            fp[class_id] += 1

        elif actual and not predicted:
            fn[class_id] += 1

# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("IMAGE-LEVEL ERROR ANALYSIS")
print("=" * 80)

print(
    f"\n{'Class':<28}"
    f"{'GT Images':>12}"
    f"{'Pred Images':>14}"
    f"{'TP':>8}"
    f"{'FP':>8}"
    f"{'FN':>8}"
)

print("-" * 80)

for class_id in range(len(CLASS_NAMES)):

    print(
        f"{CLASS_NAMES[class_id]:<28}"
        f"{ground_truth_counts[class_id]:>12}"
        f"{prediction_counts[class_id]:>14}"
        f"{tp[class_id]:>8}"
        f"{fp[class_id]:>8}"
        f"{fn[class_id]:>8}"
    )

# ------------------------------------------------------------
# Image-level recall
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("IMAGE-LEVEL RECALL")
print("=" * 80)

for class_id in range(len(CLASS_NAMES)):

    denominator = (
        tp[class_id] +
        fn[class_id]
    )

    if denominator > 0:

        recall = (
            tp[class_id] /
            denominator
        )

        print(
            f"{CLASS_NAMES[class_id]:<28}"
            f"{recall:.3f}"
        )

    else:

        print(
            f"{CLASS_NAMES[class_id]:<28}"
            f"N/A"
        )

print("\n" + "=" * 80)
print("STAGE 7.1 COMPLETE")
print("=" * 80)

print("✓ Test images analyzed")
print("✓ Ground truth compared with predictions")
print("✓ False positives and false negatives summarized")
print("✓ NO TRAINING PERFORMED")
print("✓ NO DATASET FILES MODIFIED")

In [ ]:
# ============================================================
# Stage 7.2 — YOLOv8 Confusion Matrix
#
# Generates confusion matrix on the TEST set.
# NO TRAINING
# ============================================================

from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display, Image

MODEL_PATH = (
    "/content/solar_panel_runs/"
    "yolov8n_seg_baseline/weights/best.pt"
)

DATA_YAML = (
    "/content/Solar_Thermal_Anomaly_Segmentation/"
    "data.yaml"
)

OUTPUT_DIR = Path(
    "/content/solar_panel_runs/"
    "yolov8n_seg_confusion_matrix"
)

print("=" * 80)
print("STAGE 7.2 — TEST SET CONFUSION MATRIX")
print("=" * 80)

# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

model = YOLO(MODEL_PATH)

print("\n✓ Best model loaded")

# ------------------------------------------------------------
# Generate validation/test plots
# ------------------------------------------------------------

results = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project="/content/solar_panel_runs",
    name="yolov8n_seg_confusion_matrix",
    exist_ok=True,
    verbose=False
)

print("\n✓ Test evaluation completed")

print(
    "\nResults saved to:"
)

print(
    OUTPUT_DIR
)

# ------------------------------------------------------------
# Look for confusion matrix files
# ------------------------------------------------------------

possible_files = [
    OUTPUT_DIR / "confusion_matrix.png",
    OUTPUT_DIR / "confusion_matrix_normalized.png"
]

found_files = [
    p for p in possible_files
    if p.exists()
]

print(
    f"\nConfusion matrix files found: "
    f"{len(found_files)}"
)

for file_path in found_files:

    print(
        "\n" + "-" * 80
    )

    print(
        file_path.name
    )

    display(
        Image(
            filename=str(file_path),
            width=800
        )
    )

print("\n" + "=" * 80)
print("STAGE 7.2 COMPLETE")
print("=" * 80)

print("✓ Confusion matrix generated")
print("✓ Test set used")
print("✓ NO TRAINING PERFORMED")
print("✓ NO DATASET FILES MODIFIED")

In [ ]:
from pathlib import Path
from IPython.display import display, Image

RUN_DIR = Path("/content/solar_panel_runs/yolov8n_seg_baseline")

print("=" * 80)
print("STAGE 7.3 — TRAINING CURVES & PERFORMANCE ANALYSIS")
print("=" * 80)

results_file = RUN_DIR / "results.png"

if results_file.exists():
    print("\n✓ Training results found:")
    print(results_file)
    print("\nDisplaying training curves...\n")
    display(Image(filename=str(results_file), width=1200))
else:
    print("\n❌ results.png not found.")
    print("Expected location:")
    print(results_file)

print("\n" + "=" * 80)
print("STAGE 7.3 — CELL 1 COMPLETE")
print("=" * 80)

In [ ]:
from pathlib import Path

RUN_DIR = Path("/content/solar_panel_runs/yolov8n_seg_baseline")

print("=" * 80)
print("STAGE 7.3 — LOCATING TRAINING OUTPUTS")
print("=" * 80)

if not RUN_DIR.exists():
    print("\n❌ Training directory not found:")
    print(RUN_DIR)
else:
    print(f"\n✓ Training directory found:\n{RUN_DIR}\n")

    files = sorted(
        [p for p in RUN_DIR.rglob("*") if p.is_file()],
        key=lambda p: str(p)
    )

    print(f"Total files found: {len(files)}\n")

    for p in files:
        print(p.relative_to(RUN_DIR))

print("\n" + "=" * 80)
print("STAGE 7.3 — CELL 2 COMPLETE")
print("=" * 80)

In [ ]:
from pathlib import Path

print("=" * 80)
print("STAGE 7.3 — CHECKING CURRENT COLAB RUNTIME")
print("=" * 80)

content = Path("/content")

print("\nTop-level /content items:\n")

for item in sorted(content.iterdir(), key=lambda x: x.name.lower()):
    if item.is_dir():
        print(f"[DIR]  {item.name}")
    else:
        print(f"[FILE] {item.name}")

print("\n" + "=" * 80)
print("Checking important project paths")
print("=" * 80)

paths_to_check = [
    Path("/content/Thermal-Anomalies-2"),
    Path("/content/Solar_Thermal_Anomaly_Segmentation"),
    Path("/content/solar_panel_runs"),
    Path("/content/solar_panel_runs/yolov8n_seg_baseline/weights/best.pt"),
]

for path in paths_to_check:
    print(f"{'✓' if path.exists() else '❌'} {path}")

print("\n" + "=" * 80)
print("STAGE 7.3 — CELL 3 COMPLETE")
print("=" * 80)

In [ ]:
print("=" * 80)
print("STAGE 7.3 — VERIFYING NOTEBOOK STATE")
print("=" * 80)

print("""
The Colab runtime was reset, but the dataset is still present.

We will NOT retrain yet.

The previous notebook should contain:
✓ Dataset preparation
✓ Dataset validation
✓ Taxonomy
✓ YOLOv8 training
✓ Test evaluation
✓ Test predictions
✓ Error analysis
✓ Confusion matrix results

Please check your Colab notebook tabs/cells and confirm whether
the previous training output and Stage 6/7 outputs are still visible
above this cell.

If they are visible, we can continue without repeating those stages.

If they are NOT visible, we will restore the model by rerunning
only the necessary training cell.
""")

print("=" * 80)
print("STAGE 7.3 — CELL 4 COMPLETE")
print("=" * 80)

In [ ]:
from pathlib import Path
import yaml

DATA_YAML = Path("/content/Solar_Thermal_Anomaly_Segmentation/data.yaml")

print("=" * 80)
print("PRE-TRAINING DATASET CHECK")
print("=" * 80)

print(f"\nData YAML: {DATA_YAML}")
print(f"Exists: {DATA_YAML.exists()}")

if DATA_YAML.exists():
    with open(DATA_YAML, "r") as f:
        data = yaml.safe_load(f)

    print("\nDataset configuration:")
    print(f"Train : {data.get('train')}")
    print(f"Val   : {data.get('val')}")
    print(f"Test  : {data.get('test')}")
    print(f"Classes: {data.get('names')}")

    for split in ["train", "valid", "test"]:
        image_dir = Path("/content/Solar_Thermal_Anomaly_Segmentation") / split / "images"
        label_dir = Path("/content/Solar_Thermal_Anomaly_Segmentation") / split / "labels"

        print(
            f"\n{split}: "
            f"{len(list(image_dir.glob('*')))} images, "
            f"{len(list(label_dir.glob('*.txt')))} label files"
        )

print("\n" + "=" * 80)
print("PRE-TRAINING CHECK COMPLETE")
print("=" * 80)

### Duplicate baseline-training cell intentionally removed

The baseline model is trained in **Stage 5.3**. This later recovery cell retrained the same baseline a second time and has been disabled to avoid wasting Colab GPU time. If the runtime resets and the model weights are lost, rerun Stage 5.3 after confirming the cleaned dataset is still available.

In [ ]:
from pathlib import Path
from IPython.display import display, Image

RUN_DIR = Path("/content/solar_panel_runs/yolov8n_seg_baseline")
RESULTS_PLOT = RUN_DIR / "results.png"

print("=" * 80)
print("STAGE 7.3 — TRAINING CURVES")
print("=" * 80)

if RESULTS_PLOT.exists():
    print("\n✓ results.png found")
    print(f"Path: {RESULTS_PLOT}")
    print("\nDisplaying training curves...\n")
    display(Image(filename=str(RESULTS_PLOT), width=1200))
else:
    print("\n❌ results.png not found")

print("\n" + "=" * 80)
print("STAGE 7.3 — COMPLETE")
print("=" * 80)

In [ ]:
from ultralytics import YOLO
from pathlib import Path

MODEL_PATH = "/content/solar_panel_runs/yolov8n_seg_baseline/weights/best.pt"
DATA_YAML = "/content/Solar_Thermal_Anomaly_Segmentation/data.yaml"

print("=" * 80)
print("STAGE 8.1 — BASELINE RESULT SUMMARY")
print("=" * 80)

model = YOLO(MODEL_PATH)

results = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=False,
    verbose=False
)

print("\n" + "-" * 80)
print("TEST SET OVERALL RESULTS")
print("-" * 80)

print(f"Box Precision       : {results.box.mp:.3f}")
print(f"Box Recall          : {results.box.mr:.3f}")
print(f"Box mAP@50          : {results.box.map50:.3f}")
print(f"Box mAP@50-95       : {results.box.map:.3f}")

print(f"\nMask Precision      : {results.seg.mp:.3f}")
print(f"Mask Recall         : {results.seg.mr:.3f}")
print(f"Mask mAP@50         : {results.seg.map50:.3f}")
print(f"Mask mAP@50-95      : {results.seg.map:.3f}")

print("\n" + "-" * 80)
print("PER-CLASS RESULTS")
print("-" * 80)

class_names = {
    0: "Hotspot",
    1: "Diode Failure",
    2: "Reverse Polarity Failure",
    3: "String Outage",
    4: "Shading"
}

for i, name in class_names.items():
    if i < len(results.box.ap50):
        print(f"\n{name}")
        print(f"  Box mAP@50     : {results.box.ap50[i]:.3f}")
        print(f"  Box mAP@50-95  : {results.box.ap[i]:.3f}")
        print(f"  Mask mAP@50    : {results.seg.ap50[i]:.3f}")
        print(f"  Mask mAP@50-95 : {results.seg.ap[i]:.3f}")

print("\n" + "=" * 80)
print("STAGE 8.1 COMPLETE")
print("=" * 80)

In [ ]:
print("=" * 80)
print("STAGE 8.2 — FINAL BASELINE METRICS TABLE")
print("=" * 80)

print("""
SOLAR PANEL THERMAL ANOMALY DETECTION — YOLOv8n-seg BASELINE
Test Set: 100 images | 299 instances

+--------------------------+------------+------------+
| Metric                   | Box        | Mask       |
+--------------------------+------------+------------+
| Precision                | 0.456      | 0.613      |
| Recall                   | 0.687      | 0.407      |
| mAP@50                   | 0.583      | 0.413      |
| mAP@50-95                | 0.316      | 0.203      |
+--------------------------+------------+------------+

PER-CLASS mAP@50

+--------------------------+------------+------------+
| Class                    | Box        | Mask       |
+--------------------------+------------+------------+
| Hotspot                  | 0.466      | 0.273      |
| Diode Failure            | 0.743      | 0.346      |
| Reverse Polarity        | 0.428      | 0.428      |
| String Outage            | 0.695      | 0.606      |
| Shading                  | N/A        | N/A        |
+--------------------------+------------+------------+
""")

print("=" * 80)
print("KEY BASELINE OBSERVATIONS")
print("=" * 80)

print("""
1. Box detection recall is higher than mask recall.
2. Diode Failure has the highest box mAP@50 among evaluated classes.
3. String Outage has the highest mask mAP@50 among evaluated classes.
4. Hotspot segmentation remains a challenging part of the baseline.
5. Reverse Polarity Failure has very few test instances.
6. Shading has no test instances, so it cannot be evaluated on this test split.
7. These results form the baseline for any later improvement experiment.
""")

print("=" * 80)
print("STAGE 8.2 COMPLETE")
print("=" * 80)

In [ ]:
print("=" * 80)
print("STAGE 8.3 — BASELINE FINDINGS & LIMITATIONS")
print("=" * 80)

findings = [
    "1. The YOLOv8n-seg baseline successfully performs both defect detection and segmentation on the thermal anomaly dataset.",

    "2. Detection performance is stronger than segmentation performance overall: "
    "box mAP@50 = 0.583 compared with mask mAP@50 = 0.413.",

    "3. The model achieves a box recall of 0.687, indicating that many ground-truth defect instances "
    "are detected, while mask recall of 0.407 indicates difficulty in accurately covering the complete defect regions.",

    "4. Diode Failure achieves the highest box mAP@50 among the evaluated classes (0.743).",

    "5. String Outage achieves the highest mask mAP@50 among the evaluated classes (0.606).",

    "6. Hotspot is a challenging class for precise segmentation, with mask mAP@50 = 0.273 "
    "and mask mAP@50-95 = 0.113.",

    "7. Reverse Polarity Failure has only 4 test instances, so its reported metrics should be interpreted cautiously.",

    "8. Shading has no test instances in the final test split, so quantitative test performance for this class "
    "cannot be established from the current split.",

    "9. The dataset contains substantial class imbalance, with Diode Failure having far more annotations "
    "than several other defect classes.",

    "10. The baseline provides a measurable reference point for evaluating any future model improvement."
]

print("\nBASELINE FINDINGS\n")
for item in findings:
    print(item)

print("\n" + "-" * 80)
print("MAIN LIMITATIONS")
print("-" * 80)

limitations = [
    "• Segmentation performance is lower than detection performance.",
    "• Hotspot localization/segmentation remains difficult.",
    "• Several classes have relatively few test instances.",
    "• Shading cannot be quantitatively evaluated on the current test split.",
    "• Strong class imbalance may affect performance across defect categories.",
    "• Small test populations make some per-class metrics less stable."
]

for item in limitations:
    print(item)

print("\n" + "=" * 80)
print("STAGE 8.3 COMPLETE")
print("=" * 80)

In [ ]:
import torch
from pathlib import Path

print("=" * 80)
print("STAGE 9.1 — IMPROVEMENT EXPERIMENT SETUP CHECK")
print("=" * 80)

print(f"\nGPU available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU device    : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version  : {torch.version.cuda}")

MODEL_PATH = Path("/content/solar_panel_runs/yolov8n_seg_baseline/weights/best.pt")
DATA_YAML = Path("/content/Solar_Thermal_Anomaly_Segmentation/data.yaml")

print(f"\nBaseline model exists : {MODEL_PATH.exists()}")
print(f"Dataset YAML exists   : {DATA_YAML.exists()}")

if MODEL_PATH.exists():
    print(f"Baseline model size   : {MODEL_PATH.stat().st_size / (1024**2):.2f} MB")

print("\n" + "=" * 80)
print("STAGE 9.1 COMPLETE")
print("=" * 80)

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import torch

print("=" * 80)
print("STAGE 9.2 — YOLOv8s-SEG IMPROVEMENT TRAINING")
print("=" * 80)

DATA_YAML = "/content/Solar_Thermal_Anomaly_Segmentation/data.yaml"

RUN_DIR = Path("/content/solar_panel_runs")
RUN_NAME = "yolov8s_seg_improved"

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Training device:", DEVICE)
print("\nLoading YOLOv8s-seg pretrained model...")
model = YOLO("yolov8s-seg.pt")

print("\nStarting improvement training...")
print("Dataset      :", DATA_YAML)
print("Model        : YOLOv8s-seg")
print("Image size   : 640")
print("Max epochs   : 50")
print("Batch size   : 16")
print("Early stop   : patience=10")

model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    device=DEVICE,
    workers=2,
    pretrained=True,
    project=str(RUN_DIR),
    name=RUN_NAME,
    exist_ok=True,
    patience=10,
    save=True,
    plots=True,
    verbose=True
)

BEST_MODEL = RUN_DIR / RUN_NAME / "weights" / "best.pt"

print("\n" + "=" * 80)
print("STAGE 9.2 — TRAINING COMPLETE")
print("=" * 80)

print(f"\nBest model path: {BEST_MODEL}")
print(f"Best model exists: {BEST_MODEL.exists()}")

print("\nThe improved model is ready for independent test evaluation.")

In [ ]:
from ultralytics import YOLO
from pathlib import Path

print("=" * 80)
print("STAGE 9.3 — YOLOv8s-SEG TEST EVALUATION")
print("=" * 80)

MODEL_PATH = "/content/solar_panel_runs/yolov8s_seg_improved/weights/best.pt"
DATA_YAML = "/content/Solar_Thermal_Anomaly_Segmentation/data.yaml"

model = YOLO(MODEL_PATH)

print("\nEvaluating YOLOv8s-seg on the SAME test set used for YOLOv8n...")
print("Test images : 100")
print("Image size  : 640")

results = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project="/content/solar_panel_runs",
    name="yolov8s_seg_test",
    exist_ok=True,
    verbose=True
)

print("\n" + "-" * 80)
print("YOLOv8s-SEG TEST RESULTS")
print("-" * 80)

print(f"Box Precision       : {results.box.mp:.3f}")
print(f"Box Recall          : {results.box.mr:.3f}")
print(f"Box mAP@50          : {results.box.map50:.3f}")
print(f"Box mAP@50-95       : {results.box.map:.3f}")

print(f"\nMask Precision      : {results.seg.mp:.3f}")
print(f"Mask Recall         : {results.seg.mr:.3f}")
print(f"Mask mAP@50         : {results.seg.map50:.3f}")
print(f"Mask mAP@50-95      : {results.seg.map:.3f}")

print("\n" + "=" * 80)
print("STAGE 9.3 COMPLETE")
print("=" * 80)

In [ ]:
print("=" * 80)
print("STAGE 9.4 — YOLOv8n vs YOLOv8s COMPARISON")
print("=" * 80)

baseline = {
    "Box Precision": 0.456,
    "Box Recall": 0.687,
    "Box mAP@50": 0.583,
    "Box mAP@50-95": 0.316,
    "Mask Precision": 0.613,
    "Mask Recall": 0.407,
    "Mask mAP@50": 0.413,
    "Mask mAP@50-95": 0.203,
}

improved = {
    "Box Precision": 0.575,
    "Box Recall": 0.627,
    "Box mAP@50": 0.610,
    "Box mAP@50-95": 0.347,
    "Mask Precision": 0.466,
    "Mask Recall": 0.488,
    "Mask mAP@50": 0.436,
    "Mask mAP@50-95": 0.211,
}

print("\nTEST SET COMPARISON")
print("-" * 80)
print(f"{'Metric':<25}{'YOLOv8n':>12}{'YOLOv8s':>12}{'Change':>12}")
print("-" * 80)

for metric in baseline:
    change = improved[metric] - baseline[metric]
    sign = "+" if change >= 0 else ""
    print(
        f"{metric:<25}"
        f"{baseline[metric]:>12.3f}"
        f"{improved[metric]:>12.3f}"
        f"{sign}{change:>11.3f}"
    )

print("-" * 80)

print("\nPER-CLASS mAP@50")
print("-" * 80)
print(f"{'Class':<28}{'Box n':>10}{'Box s':>10}{'Mask n':>10}{'Mask s':>10}")
print("-" * 80)

classes = {
    "Hotspot": (0.466, 0.512, 0.273, 0.324),
    "Diode Failure": (0.743, 0.769, 0.346, 0.377),
    "Reverse Polarity Failure": (0.428, 0.383, 0.428, 0.383),
    "String Outage": (0.695, 0.778, 0.606, 0.662),
}

for name, values in classes.items():
    print(
        f"{name:<28}"
        f"{values[0]:>10.3f}"
        f"{values[1]:>10.3f}"
        f"{values[2]:>10.3f}"
        f"{values[3]:>10.3f}"
    )

print("-" * 80)
print("Shading: N/A on the test set for both models")

print("\n" + "=" * 80)
print("STAGE 9.4 COMPLETE")
print("=" * 80)

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import random

print("=" * 80)
print("STAGE 9.5 — VISUAL PREDICTION COMPARISON")
print("=" * 80)

DATA_DIR = Path("/content/Solar_Thermal_Anomaly_Segmentation/test/images")

BASELINE_MODEL = "/content/solar_panel_runs/yolov8n_seg_baseline/weights/best.pt"
IMPROVED_MODEL = "/content/solar_panel_runs/yolov8s_seg_improved/weights/best.pt"

OUTPUT_DIR = Path("/content/solar_panel_runs/model_comparison_predictions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

images = sorted([
    p for p in DATA_DIR.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
])

random.seed(42)
selected_images = random.sample(images, min(6, len(images)))

print(f"\nTest images available : {len(images)}")
print(f"Images selected       : {len(selected_images)}")

print("\nSelected images:")
for img in selected_images:
    print(" -", img.name)

baseline_model = YOLO(BASELINE_MODEL)
improved_model = YOLO(IMPROVED_MODEL)

print("\nGenerating YOLOv8n predictions...")

baseline_model.predict(
    source=[str(p) for p in selected_images],
    imgsz=640,
    conf=0.25,
    device=0,
    save=True,
    project=str(OUTPUT_DIR),
    name="yolov8n",
    exist_ok=True,
    verbose=False
)

print("✓ YOLOv8n predictions saved")

print("\nGenerating YOLOv8s predictions...")

improved_model.predict(
    source=[str(p) for p in selected_images],
    imgsz=640,
    conf=0.25,
    device=0,
    save=True,
    project=str(OUTPUT_DIR),
    name="yolov8s",
    exist_ok=True,
    verbose=False
)

print("✓ YOLOv8s predictions saved")

print("\n" + "-" * 80)
print("OUTPUT DIRECTORIES")
print("-" * 80)

print("YOLOv8n:", OUTPUT_DIR / "yolov8n")
print("YOLOv8s:", OUTPUT_DIR / "yolov8s")

print("\n" + "=" * 80)
print("STAGE 9.5 COMPLETE")
print("=" * 80)

In [ ]:
print("=" * 80)
print("STAGE 9.6 — FINAL IMPROVEMENT EXPERIMENT ANALYSIS")
print("=" * 80)

baseline = {
    "Box Precision": 0.456,
    "Box Recall": 0.687,
    "Box mAP@50": 0.583,
    "Box mAP@50-95": 0.316,
    "Mask Precision": 0.613,
    "Mask Recall": 0.407,
    "Mask mAP@50": 0.413,
    "Mask mAP@50-95": 0.203,
}

improved = {
    "Box Precision": 0.575,
    "Box Recall": 0.627,
    "Box mAP@50": 0.610,
    "Box mAP@50-95": 0.347,
    "Mask Precision": 0.466,
    "Mask Recall": 0.488,
    "Mask mAP@50": 0.436,
    "Mask mAP@50-95": 0.211,
}

print("\n1. EXPERIMENT SETUP")
print("-" * 80)
print("Baseline model : YOLOv8n-seg")
print("Improved model : YOLOv8s-seg")
print("Dataset        : Same clean thermal anomaly segmentation dataset")
print("Test set       : Same 100 test images")
print("Image size     : 640 x 640")
print("Classes        : 5")
print("Comparison     : Detection boxes + segmentation masks")

print("\n2. OVERALL TEST RESULTS")
print("-" * 80)

for metric in baseline:
    change = improved[metric] - baseline[metric]
    print(
        f"{metric:<25} "
        f"YOLOv8n={baseline[metric]:.3f}  "
        f"YOLOv8s={improved[metric]:.3f}  "
        f"Change={change:+.3f}"
    )

print("\n3. OBSERVED IMPROVEMENTS")
print("-" * 80)

print("• Box mAP@50 increased from 0.583 to 0.610.")
print("• Box mAP@50-95 increased from 0.316 to 0.347.")
print("• Mask mAP@50 increased from 0.413 to 0.436.")
print("• Mask mAP@50-95 increased from 0.203 to 0.211.")
print("• Mask recall increased from 0.407 to 0.488.")
print("• Box precision increased from 0.456 to 0.575.")

print("\n4. TRADE-OFFS")
print("-" * 80)

print("• Box recall decreased from 0.687 to 0.627.")
print("• Mask precision decreased from 0.613 to 0.466.")
print("• Therefore, the model change altered the precision-recall balance.")

print("\n5. PER-CLASS mAP@50 CHANGES")
print("-" * 80)

per_class = {
    "Hotspot":        (0.466, 0.512, 0.273, 0.324),
    "Diode Failure":  (0.743, 0.769, 0.346, 0.377),
    "Reverse Polarity Failure": (0.428, 0.383, 0.428, 0.383),
    "String Outage":  (0.695, 0.778, 0.606, 0.662),
}

for cls, (bn, bs, mn, ms) in per_class.items():
    print(
        f"{cls:<28} "
        f"Box: {bn:.3f} -> {bs:.3f}  "
        f"Mask: {mn:.3f} -> {ms:.3f}"
    )

print("\n6. DATASET LIMITATION")
print("-" * 80)
print("• Shading has no test instances, so no test-set metric is available.")
print("• Reverse Polarity Failure has only 4 test instances.")
print("• Results for small classes should therefore be interpreted cautiously.")

print("\n7. FINAL EXPERIMENT SUMMARY")
print("-" * 80)
print(
    "The YOLOv8s-seg experiment produced higher overall box and mask "
    "mAP values on the same test set, while also changing the precision-"
    "recall balance. The experiment demonstrates the effect of increasing "
    "model capacity, but the results are specific to this dataset and test split."
)

print("\n" + "=" * 80)
print("STAGE 9.6 COMPLETE")
print("=" * 80)

In [ ]:
from pathlib import Path

print("=" * 80)
print("STAGE 10.1 — GITHUB PROJECT FILE CHECK")
print("=" * 80)

BASE = Path("/content/solar_panel_runs")

important_paths = {
    "Baseline model": BASE / "yolov8n_seg_baseline/weights/best.pt",
    "Improved model": BASE / "yolov8s_seg_improved/weights/best.pt",
    "Baseline training results": BASE / "yolov8n_seg_baseline/results.png",
    "Improved training results": BASE / "yolov8s_seg_improved/results.png",
    "Baseline test results": BASE / "yolov8n_seg_test",
    "Improved test results": BASE / "yolov8s_seg_test",
    "Visual comparison": BASE / "model_comparison_predictions",
    "Clean dataset": Path("/content/Solar_Thermal_Anomaly_Segmentation"),
    "Dataset YAML": Path("/content/Solar_Thermal_Anomaly_Segmentation/data.yaml"),
    "Original dataset": Path("/content/Thermal-Anomalies-2"),
    "YOLOv8n pretrained model": Path("/content/yolov8n-seg.pt"),
    "YOLOv8s pretrained model": Path("/content/yolov8s-seg.pt"),
}

print("\nIMPORTANT PROJECT FILES")
print("-" * 80)

for name, path in important_paths.items():
    status = "✓ EXISTS" if path.exists() else "✗ MISSING"
    print(f"{status:<12} {name}")
    print(f"             {path}")

print("\n" + "-" * 80)

if BASE.exists():
    print("\nsolar_panel_runs contents:")
    for item in sorted(BASE.iterdir()):
        print(" -", item.name)

print("\n" + "=" * 80)
print("STAGE 10.1 COMPLETE")
print("=" * 80)

In [ ]:
from pathlib import Path

GITHUB_DIR = Path("/content/Solar-Panel-Thermal-Anomaly-Detection")

folders = [
    "results",
    "results/training",
    "results/evaluation",
    "results/predictions",
    "models",
    "references",
    "poster",
]

for folder in folders:
    (GITHUB_DIR / folder).mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("STAGE 10.2 — GITHUB PROJECT STRUCTURE")
print("=" * 80)

print(f"\nGitHub project directory:")
print(GITHUB_DIR)

print("\nCreated structure:")
print("""
Solar-Panel-Thermal-Anomaly-Detection/
│
├── results/
│   ├── training/
│   ├── evaluation/
│   └── predictions/
│
├── models/
│
├── references/
│
├── poster/
│
└── README.md          ← will be created later
""")

print("\n✓ GitHub project folders created")
print("\n" + "=" * 80)
print("STAGE 10.2 COMPLETE")
print("=" * 80)

In [ ]:
from pathlib import Path

print("=" * 80)
print("STAGE 10.3 — LOCATE PROJECT NOTEBOOK")
print("=" * 80)

search_locations = [
    Path("/content"),
    Path("/content/drive/MyDrive"),
]

notebooks = []

for location in search_locations:
    if location.exists():
        for nb in location.rglob("*.ipynb"):
            if nb not in notebooks:
                notebooks.append(nb)

print("\nNotebook files found:")
print("-" * 80)

if notebooks:
    for i, nb in enumerate(sorted(notebooks), 1):
        print(f"{i}. {nb}")
else:
    print("No .ipynb files found in the searched locations.")

print("\n" + "=" * 80)
print("STAGE 10.3 COMPLETE")
print("=" * 80)

In [ ]:
from pathlib import Path
import os

print("=" * 80)
print("STAGE 10.4 — COLAB NOTEBOOK ENVIRONMENT CHECK")
print("=" * 80)

print("\nCURRENT WORKING DIRECTORY")
print("-" * 80)
print(os.getcwd())

print("\n/CONTENT ROOT")
print("-" * 80)

content = Path("/content")

for item in sorted(content.iterdir()):
    if item.is_dir():
        print(f"[DIR]  {item.name}")
    else:
        print(f"[FILE] {item.name}")

print("\nPOSSIBLE NOTEBOOK-RELATED FILES")
print("-" * 80)

patterns = [
    "*.ipynb",
    "*notebook*",
    "*Notebook*",
]

found = []

for pattern in patterns:
    for item in content.rglob(pattern):
        if item not in found:
            found.append(item)

if found:
    for item in sorted(found):
        print(item)
else:
    print("No notebook-related files found.")

print("\n" + "=" * 80)
print("STAGE 10.4 COMPLETE")
print("=" * 80)

In [ ]:
from pathlib import Path
import shutil

PROJECT = Path("/content/Solar-Panel-Thermal-Anomaly-Detection")
RUNS = Path("/content/solar_panel_runs")

print("=" * 80)
print("STAGE 10.5 — COPY PROJECT RESULTS")
print("=" * 80)

# ------------------------------------------------------------------
# 1. Copy training curves
# ------------------------------------------------------------------

training_sources = {
    RUNS / "yolov8n_seg_baseline/results.png":
        PROJECT / "results/training/yolov8n_baseline_results.png",

    RUNS / "yolov8s_seg_improved/results.png":
        PROJECT / "results/training/yolov8s_improved_results.png",
}

print("\nTRAINING RESULTS")
print("-" * 80)

for source, destination in training_sources.items():
    if source.exists():
        shutil.copy2(source, destination)
        print("✓ Copied:", source.name)
    else:
        print("✗ Missing:", source)


# ------------------------------------------------------------------
# 2. Copy improved model evaluation results
# ------------------------------------------------------------------

evaluation_source = RUNS / "yolov8s_seg_test"
evaluation_destination = PROJECT / "results/evaluation/yolov8s_test"

print("\nIMPROVED MODEL EVALUATION")
print("-" * 80)

if evaluation_source.exists():
    if evaluation_destination.exists():
        shutil.rmtree(evaluation_destination)

    shutil.copytree(evaluation_source, evaluation_destination)
    print("✓ Copied YOLOv8s test evaluation results")
else:
    print("✗ YOLOv8s test results not found")


# ------------------------------------------------------------------
# 3. Copy visual comparison predictions
# ------------------------------------------------------------------

prediction_source = RUNS / "model_comparison_predictions"
prediction_destination = PROJECT / "results/predictions/model_comparison"

print("\nVISUAL PREDICTIONS")
print("-" * 80)

if prediction_source.exists():
    if prediction_destination.exists():
        shutil.rmtree(prediction_destination)

    shutil.copytree(prediction_source, prediction_destination)
    print("✓ Copied model comparison predictions")
else:
    print("✗ Model comparison predictions not found")


# ------------------------------------------------------------------
# 4. Create a metrics summary text file
# ------------------------------------------------------------------

metrics_file = PROJECT / "results/evaluation/model_metrics.txt"

metrics_text = """SOLAR PANEL THERMAL ANOMALY DETECTION
MODEL COMPARISON RESULTS

Dataset:
Thermal Anomalies — Solar Panel Data Set

Final taxonomy:
0 Hotspot
1 Diode Failure
2 Reverse Polarity Failure
3 String Outage
4 Shading

TEST SET:
100 images

YOLOv8n-seg BASELINE
--------------------
Box Precision: 0.456
Box Recall: 0.687
Box mAP@50: 0.583
Box mAP@50-95: 0.316

Mask Precision: 0.613
Mask Recall: 0.407
Mask mAP@50: 0.413
Mask mAP@50-95: 0.203


YOLOv8s-seg IMPROVED
--------------------
Box Precision: 0.575
Box Recall: 0.627
Box mAP@50: 0.610
Box mAP@50-95: 0.347

Mask Precision: 0.466
Mask Recall: 0.488
Mask mAP@50: 0.436
Mask mAP@50-95: 0.211


CHANGES: YOLOv8n -> YOLOv8s
---------------------------
Box Precision: +0.119
Box Recall: -0.060
Box mAP@50: +0.027
Box mAP@50-95: +0.031

Mask Precision: -0.147
Mask Recall: +0.081
Mask mAP@50: +0.023
Mask mAP@50-95: +0.008


PER-CLASS mAP@50
----------------
Hotspot:
  Box: 0.466 -> 0.512
  Mask: 0.273 -> 0.324

Diode Failure:
  Box: 0.743 -> 0.769
  Mask: 0.346 -> 0.377

Reverse Polarity Failure:
  Box: 0.428 -> 0.383
  Mask: 0.428 -> 0.383

String Outage:
  Box: 0.695 -> 0.778
  Mask: 0.606 -> 0.662

Shading:
  N/A — no test instances


IMPORTANT DATASET LIMITATIONS
-----------------------------
Shading has no test instances.
Reverse Polarity Failure has only 4 test instances.
Results for small classes should be interpreted cautiously.
"""

metrics_file.write_text(metrics_text, encoding="utf-8")

print("\nMETRICS SUMMARY")
print("-" * 80)
print("✓ Created:", metrics_file)


# ------------------------------------------------------------------
# 5. Display resulting structure
# ------------------------------------------------------------------

print("\nGITHUB PROJECT CONTENTS")
print("-" * 80)

for path in sorted(PROJECT.rglob("*")):
    if path.is_file():
        relative = path.relative_to(PROJECT)
        size_kb = path.stat().st_size / 1024
        print(f"{relative}  ({size_kb:.1f} KB)")

print("\n" + "=" * 80)
print("STAGE 10.5 COMPLETE")
print("=" * 80)

In [ ]:
from pathlib import Path

print("=" * 80)
print("STAGE 10.6 — GOOGLE DRIVE CHECK")
print("=" * 80)

drive = Path("/content/drive")

if drive.exists():
    print("\n✓ /content/drive exists")

    mydrive = drive / "MyDrive"

    if mydrive.exists():
        print("✓ MyDrive exists")

        print("\nTop-level MyDrive contents:")
        print("-" * 80)

        for item in sorted(mydrive.iterdir()):
            print(
                f"[DIR]  {item.name}" if item.is_dir()
                else f"[FILE] {item.name}"
            )

        print("\nSearching MyDrive for notebooks...")
        print("-" * 80)

        notebooks = list(mydrive.rglob("*.ipynb"))

        if notebooks:
            for nb in notebooks:
                print(nb)
        else:
            print("No .ipynb files found in MyDrive.")

    else:
        print("✗ MyDrive is not available.")
else:
    print("✗ Google Drive is not mounted.")

print("\n" + "=" * 80)
print("STAGE 10.6 COMPLETE")
print("=" * 80)

In [ ]:
from pathlib import Path

print("=" * 80)
print("STAGE 10.7 — FINAL NOTEBOOK RECOVERY CHECK")
print("=" * 80)

locations = [
    Path("/mnt/data"),
    Path("/content"),
]

for location in locations:
    print(f"\nChecking: {location}")
    print("-" * 80)

    if not location.exists():
        print("Location does not exist.")
        continue

    files = list(location.rglob("*"))

    notebook_files = [
        f for f in files
        if f.is_file() and f.suffix.lower() == ".ipynb"
    ]

    if notebook_files:
        for nb in notebook_files:
            print("✓", nb)
    else:
        print("No .ipynb files found.")

print("\n" + "=" * 80)
print("STAGE 10.7 COMPLETE")
print("=" * 80)

In [ ]:
from pathlib import Path
import json

PROJECT = Path("/content/Solar-Panel-Thermal-Anomaly-Detection")
NOTEBOOK = PROJECT / "Solar_Panel_Thermal_Anomaly_Detection.ipynb"

def make_md(text):
    return {
        "cell_type": "markdown",
        "metadata": {},
        "source": [text]
    }

def make_code(text):
    return {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [text]
    }

cells = []

cells.append(make_md("# Solar Panel Thermal Anomaly Detection Using YOLOv8\n\n"
                     "## Deep Learning Mini Project\n\n"
                     "Thermal-image based detection and segmentation of solar-panel anomalies.\n\n"
                     "**Final classes:** Hotspot, Diode Failure, Reverse Polarity Failure, String Outage, Shading."))

cells.append(make_md("## 1. Problem Statement\n\n"
                     "Solar panels can develop faults such as hotspots, diode failures, "
                     "reverse polarity failures, string outages, and shading-related anomalies.\n\n"
                     "The objective of this project is to use deep learning to detect and "
                     "segment thermal anomalies in solar-panel images."))

cells.append(make_md("## 2. Dataset\n\n"
                     "**Dataset:** Thermal Anomalies — Solar Panel Data Set\n\n"
                     "Original dataset statistics:\n\n"
                     "- 1013 images\n"
                     "- 711 training images\n"
                     "- 202 validation images\n"
                     "- 100 test images\n"
                     "- 17 original classes\n\n"
                     "The dataset contained both bounding-box and polygon annotations."))

cells.append(make_md("## 3. Final Taxonomy\n\n"
                     "| ID | Class |\n"
                     "|---:|---|\n"
                     "| 0 | Hotspot |\n"
                     "| 1 | Diode Failure |\n"
                     "| 2 | Reverse Polarity Failure |\n"
                     "| 3 | String Outage |\n"
                     "| 4 | Shading |\n\n"
                     "Related hotspot variants were merged into the Hotspot class. "
                     "Several very low-frequency or unsuitable original categories were excluded."))

cells.append(make_md("## 4. Clean Segmentation Dataset\n\n"
                     "Clean dataset location:\n\n"
                     "`/content/Solar_Thermal_Anomaly_Segmentation`\n\n"
                     "Statistics:\n\n"
                     "- Images: 1013\n"
                     "- Train: 711\n"
                     "- Validation: 202\n"
                     "- Test: 100\n"
                     "- Classes: 5\n"
                     "- Output annotations: 3127\n\n"
                     "Bounding-box annotations for selected classes were converted to rectangular "
                     "segmentation polygons, while existing polygon annotations were retained."))

cells.append(make_md("## 5. Dataset Validation\n\n"
                     "The clean dataset was checked for missing images, corrupt images, invalid "
                     "class IDs, invalid polygon coordinates, zero-area polygons, and duplicate annotations.\n\n"
                     "No corrupted images or invalid annotations were found.\n\n"
                     "The 60 images without selected-class labels were investigated and explained "
                     "by excluded classes or empty original labels."))

cells.append(make_md("## 6. YOLOv8n Baseline\n\n"
                     "Baseline model: **YOLOv8n-seg**\n\n"
                     "Training configuration:\n\n"
                     "- Image size: 640\n"
                     "- Batch size: 16\n"
                     "- Maximum epochs: 50\n"
                     "- GPU: Tesla T4\n"
                     "- Early stopping patience: 10\n\n"
                     "Training stopped at epoch 48. Best checkpoint occurred at epoch 38."))

cells.append(make_md("## 7. YOLOv8n Baseline Test Results\n\n"
                     "| Metric | Result |\n"
                     "|---|---:|\n"
                     "| Box Precision | 0.456 |\n"
                     "| Box Recall | 0.687 |\n"
                     "| Box mAP@50 | 0.583 |\n"
                     "| Box mAP@50-95 | 0.316 |\n"
                     "| Mask Precision | 0.613 |\n"
                     "| Mask Recall | 0.407 |\n"
                     "| Mask mAP@50 | 0.413 |\n"
                     "| Mask mAP@50-95 | 0.203 |\n\n"
                     "The test set contained no Shading instances."))

cells.append(make_md("## 8. Error Analysis\n\n"
                     "Baseline image-level analysis showed that Hotspot recognition was stronger "
                     "than precise hotspot mask localization.\n\n"
                     "Smaller classes had very few test instances, so their metrics should be "
                     "interpreted cautiously."))

cells.append(make_md("## 9. Improvement Experiment\n\n"
                     "Improved model: **YOLOv8s-seg**\n\n"
                     "The same dataset, image size, batch size, and test set were used for the "
                     "comparison.\n\n"
                     "The model trained for 50 epochs, with the best validation checkpoint at epoch 42."))

cells.append(make_md("## 10. YOLOv8n vs YOLOv8s\n\n"
                     "| Metric | YOLOv8n | YOLOv8s | Change |\n"
                     "|---|---:|---:|---:|\n"
                     "| Box Precision | 0.456 | 0.575 | +0.119 |\n"
                     "| Box Recall | 0.687 | 0.627 | -0.060 |\n"
                     "| Box mAP@50 | 0.583 | 0.610 | +0.027 |\n"
                     "| Box mAP@50-95 | 0.316 | 0.347 | +0.031 |\n"
                     "| Mask Precision | 0.613 | 0.466 | -0.147 |\n"
                     "| Mask Recall | 0.407 | 0.488 | +0.081 |\n"
                     "| Mask mAP@50 | 0.413 | 0.436 | +0.023 |\n"
                     "| Mask mAP@50-95 | 0.203 | 0.211 | +0.008 |"))

cells.append(make_md("## 11. Limitations\n\n"
                     "1. The dataset is strongly class-imbalanced.\n"
                     "2. Shading has no test instances.\n"
                     "3. Reverse Polarity Failure has only 4 test instances.\n"
                     "4. Small-class metrics should be interpreted cautiously.\n"
                     "5. Precise thermal anomaly segmentation remains challenging.\n"
                     "6. Results are specific to this dataset and test split."))

cells.append(make_md("## 12. Conclusion\n\n"
                     "This project developed a YOLOv8-based thermal anomaly detection and "
                     "segmentation pipeline for solar-panel inspection.\n\n"
                     "The workflow included dataset verification, taxonomy consolidation, "
                     "segmentation preparation, validation, baseline training, evaluation, "
                     "error analysis, improvement training, and model comparison."))

cells.append(make_md("## 13. References\n\n"
                     "- Ultralytics YOLO\n"
                     "- Thermal Anomalies — Solar Panel Data Set\n"
                     "- Original dataset source used during project preparation"))

notebook = {
    "cells": cells,
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {
            "name": "python",
            "version": "3.x"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 5
}

NOTEBOOK.write_text(
    json.dumps(notebook, indent=2),
    encoding="utf-8"
)

print("=" * 80)
print("STAGE 10.8A COMPLETE")
print("=" * 80)
print("Notebook created:")
print(NOTEBOOK)
print("Cells:", len(cells))

In [ ]:
from pathlib import Path
import shutil

PROJECT = Path("/content/Solar-Panel-Thermal-Anomaly-Detection")
RUNS = Path("/content/solar_panel_runs")

print("=" * 80)
print("STAGE 10.9 — ORGANIZE MODELS AND RESULTS")
print("=" * 80)

# ------------------------------------------------------------
# Copy trained model weights
# ------------------------------------------------------------

model_files = {
    RUNS / "yolov8n_seg_baseline/weights/best.pt":
        PROJECT / "models/yolov8n_seg_baseline_best.pt",

    RUNS / "yolov8s_seg_improved/weights/best.pt":
        PROJECT / "models/yolov8s_seg_improved_best.pt",
}

print("\nMODEL WEIGHTS")
print("-" * 80)

for source, destination in model_files.items():
    if source.exists():
        shutil.copy2(source, destination)
        size_mb = destination.stat().st_size / (1024 * 1024)
        print(f"✓ {destination.name} ({size_mb:.2f} MB)")
    else:
        print(f"✗ Missing: {source}")


# ------------------------------------------------------------
# Copy YOLOv8s test evaluation
# ------------------------------------------------------------

source = RUNS / "yolov8s_seg_test"
destination = PROJECT / "results/evaluation/yolov8s_test"

if source.exists():
    print("\n✓ YOLOv8s evaluation already available")
else:
    print("\n✗ YOLOv8s evaluation folder missing")


# ------------------------------------------------------------
# Copy training curves
# ------------------------------------------------------------

print("\nTRAINING CURVES")
print("-" * 80)

for filename in [
    "yolov8n_baseline_results.png",
    "yolov8s_improved_results.png"
]:
    path = PROJECT / "results/training" / filename
    if path.exists():
        print("✓", filename)
    else:
        print("✗", filename)


# ------------------------------------------------------------
# Verify notebook
# ------------------------------------------------------------

notebook = PROJECT / "Solar_Panel_Thermal_Anomaly_Detection.ipynb"

print("\nNOTEBOOK")
print("-" * 80)

if notebook.exists():
    print("✓", notebook.name)
    print(f"  Size: {notebook.stat().st_size / 1024:.1f} KB")
else:
    print("✗ Notebook missing")


# ------------------------------------------------------------
# Final project tree
# ------------------------------------------------------------

print("\nGITHUB PROJECT TREE")
print("-" * 80)

for path in sorted(PROJECT.rglob("*")):
    if path.is_file():
        relative = path.relative_to(PROJECT)
        print(relative)

print("\n" + "=" * 80)
print("STAGE 10.9 COMPLETE")
print("=" * 80)

In [ ]:
from pathlib import Path

PROJECT = Path("/content/Solar-Panel-Thermal-Anomaly-Detection")

requirements = """ultralytics==8.4.163
"""

requirements_path = PROJECT / "requirements.txt"
requirements_path.write_text(requirements, encoding="utf-8")

print("=" * 80)
print("STAGE 10.10 — REQUIREMENTS FILE")
print("=" * 80)

print("\nCreated:")
print(requirements_path)

print("\nContents:")
print("-" * 80)
print(requirements)

print("=" * 80)
print("STAGE 10.10 COMPLETE")
print("=" * 80)

In [ ]:
from pathlib import Path

PROJECT = Path("/content/Solar-Panel-Thermal-Anomaly-Detection")
README = PROJECT / "README.md"

lines = [
    "# Solar Panel Thermal Anomaly Detection Using YOLOv8",
    "",
    "## Deep Learning Mini Project",
    "",
    "A deep learning project for detecting and segmenting thermal anomalies in solar-panel images using YOLOv8 segmentation models.",
    "",
    "The project compares a YOLOv8n-seg baseline with a YOLOv8s-seg model using the same dataset and test split.",
    "",
    "---",
    "",
    "## 1. Problem Statement",
    "",
    "Solar panels can develop faults such as hotspots, diode failures, reverse polarity failures, string outages, and shading-related anomalies.",
    "",
    "Thermal imaging can reveal abnormal temperature patterns associated with these faults.",
    "",
    "The objective of this project is to develop a deep learning pipeline that can:",
    "",
    "- Detect thermal anomalies in solar-panel images.",
    "- Segment detected anomaly regions.",
    "- Classify anomalies into defined fault categories.",
    "- Compare different YOLOv8 segmentation model configurations.",
    "",
    "---",
    "",
    "## 2. Objectives",
    "",
    "1. Prepare a clean thermal-anomaly segmentation dataset.",
    "2. Consolidate the original dataset labels into a practical five-class taxonomy.",
    "3. Train a YOLOv8n segmentation baseline.",
    "4. Evaluate detection and segmentation performance.",
    "5. Perform error analysis.",
    "6. Train a YOLOv8s segmentation model as an improvement experiment.",
    "7. Compare both models on the same test set.",
    "8. Document limitations and observations.",
    "",
    "---",
    "",
    "## 3. Dataset",
    "",
    "**Dataset:** Thermal Anomalies — Solar Panel Data Set",
    "",
    "Original dataset statistics:",
    "",
    "- Total images: 1013",
    "- Training images: 711",
    "- Validation images: 202",
    "- Test images: 100",
    "- Original classes: 17",
    "",
    "The original dataset contained both bounding-box and polygon annotations.",
    "",
    "### Clean Dataset",
    "",
    "A cleaned segmentation dataset was prepared for this project.",
    "",
    "- Images: 1013",
    "- Training: 711",
    "- Validation: 202",
    "- Test: 100",
    "- Final classes: 5",
    "- Output annotations: 3127",
    "",
    "---",
    "",
    "## 4. Final Taxonomy",
    "",
    "| ID | Final Class |",
    "|---:|---|",
    "| 0 | Hotspot |",
    "| 1 | Diode Failure |",
    "| 2 | Reverse Polarity Failure |",
    "| 3 | String Outage |",
    "| 4 | Shading |",
    "",
    "Related hotspot variants were merged into the Hotspot class.",
    "",
    "### Annotation Distribution",
    "",
    "| Class | Annotations |",
    "|---|---:|",
    "| Hotspot | 927 |",
    "| Diode Failure | 1950 |",
    "| Reverse Polarity Failure | 79 |",
    "| String Outage | 162 |",
    "| Shading | 9 |",
    "",
    "The dataset is strongly class-imbalanced, with Diode Failure having substantially more annotations than the other classes.",
    "",
    "---",
    "",
    "## 5. Dataset Preparation",
    "",
    "The original dataset was preserved separately.",
    "",
    "For the final segmentation dataset:",
    "",
    "- Selected classes were retained.",
    "- Related hotspot classes were merged.",
    "- Bounding-box annotations for selected classes were converted into rectangular segmentation polygons.",
    "- Existing polygon annotations were retained.",
    "- Excluded classes were removed from the final annotation set.",
    "- Dataset validation was performed after conversion.",
    "",
    "The clean dataset was checked for missing images, corrupt images, invalid class IDs, invalid polygon coordinates, zero-area polygons, and duplicate annotations.",
    "",
    "No corrupted images or invalid annotations were found.",
    "",
    "The 60 images without selected-class labels were investigated and explained by excluded original classes or empty original label files.",
    "",
    "---",
    "",
    "## 6. Methodology",
    "",
    "```text",
    "Thermal Solar-Panel Images",
    "          |",
    "          v",
    "Dataset Verification",
    "          |",
    "          v",
    "Taxonomy Consolidation",
    "          |",
    "          v",
    "Clean Segmentation Dataset",
    "          |",
    "          v",
    "YOLOv8 Segmentation",
    "          |",
    "          +----------------------+",
    "          |                      |",
    "          v                      v",
    "     YOLOv8n-seg            YOLOv8s-seg",
    "       Baseline             Improvement",
    "          |                      |",
    "          +----------+-----------+",
    "                     |",
    "                     v",
    "              Test Evaluation",
    "                     |",
    "                     v",
    "               Error Analysis",
    "                     |",
    "                     v",
    "                Model Comparison",
    "```",
    "",
    "---",
    "",
    "## 7. YOLOv8n Baseline",
    "",
    "**Model:** YOLOv8n-seg",
    "",
    "Training configuration:",
    "",
    "- Image size: 640",
    "- Batch size: 16",
    "- Maximum epochs: 50",
    "- Early stopping patience: 10",
    "- GPU: Tesla T4",
    "- Pretrained weights: Yes",
    "",
    "Training stopped at epoch 48. The best checkpoint occurred at epoch 38.",
    "",
    "---",
    "",
    "## 8. Baseline Test Results",
    "",
    "| Metric | YOLOv8n |",
    "|---|---:|",
    "| Box Precision | 0.456 |",
    "| Box Recall | 0.687 |",
    "| Box mAP@50 | 0.583 |",
    "| Box mAP@50-95 | 0.316 |",
    "| Mask Precision | 0.613 |",
    "| Mask Recall | 0.407 |",
    "| Mask mAP@50 | 0.413 |",
    "| Mask mAP@50-95 | 0.203 |",
    "",
    "The test set contained no Shading instances.",
    "",
    "---",
    "",
    "## 9. Error Analysis",
    "",
    "| Class | Ground-truth Images | Predicted Images | Image-level Recall |",
    "|---|---:|---:|---:|",
    "| Hotspot | 46 | 47 | 0.739 |",
    "| Diode Failure | 72 | 67 | 0.819 |",
    "| Reverse Polarity Failure | 4 | 4 | 0.500 |",
    "| String Outage | 5 | 6 | 0.800 |",
    "| Shading | 0 | 0 | N/A |",
    "",
    "Hotspot image-level recognition was stronger than precise hotspot mask localization.",
    "",
    "Small test populations for some classes require cautious interpretation.",
    "",
    "---",
    "",
    "## 10. Improvement Experiment",
    "",
    "**Model:** YOLOv8s-seg",
    "",
    "The same clean dataset and test split were used for the comparison.",
    "",
    "Configuration:",
    "",
    "- Image size: 640",
    "- Batch size: 16",
    "- Maximum epochs: 50",
    "- Early stopping patience: 10",
    "- GPU: Tesla T4",
    "",
    "The model trained for 50 epochs, with the best validation checkpoint at epoch 42.",
    "",
    "---",
    "",
    "## 11. YOLOv8n vs YOLOv8s",
    "",
    "| Metric | YOLOv8n | YOLOv8s | Change |",
    "|---|---:|---:|---:|",
    "| Box Precision | 0.456 | 0.575 | +0.119 |",
    "| Box Recall | 0.687 | 0.627 | -0.060 |",
    "| Box mAP@50 | 0.583 | 0.610 | +0.027 |",
    "| Box mAP@50-95 | 0.316 | 0.347 | +0.031 |",
    "| Mask Precision | 0.613 | 0.466 | -0.147 |",
    "| Mask Recall | 0.407 | 0.488 | +0.081 |",
    "| Mask mAP@50 | 0.413 | 0.436 | +0.023 |",
    "| Mask mAP@50-95 | 0.203 | 0.211 | +0.008 |",
    "",
    "The YOLOv8s experiment produced higher overall box and mask mAP values on this test split, while changing the precision-recall balance.",
    "",
    "---",
    "",
    "## 12. Limitations",
    "",
    "1. The dataset is strongly class-imbalanced.",
    "2. Shading has only 9 annotations and no validation/test instances.",
    "3. Reverse Polarity Failure has only 4 test instances.",
    "4. Small-class metrics should be interpreted cautiously.",
    "5. Precise thermal anomaly segmentation remains challenging.",
    "6. Results are specific to this dataset and test split.",
    "",
    "---",
    "",
    "## 13. Repository Structure",
    "",
    "```text",
    "Solar-Panel-Thermal-Anomaly-Detection/",
    "|",
    "|-- Solar_Panel_Thermal_Anomaly_Detection.ipynb",
    "|-- requirements.txt",
    "|",
    "|-- models/",
    "|   |-- yolov8n_seg_baseline_best.pt",
    "|   `-- yolov8s_seg_improved_best.pt",
    "|",
    "|-- results/",
    "|   |-- training/",
    "|   |-- evaluation/",
    "|   `-- predictions/",
    "|",
    "|-- references/",
    "`-- poster/",
    "```",
    "",
    "---",
    "",
    "## 14. Installation",
    "",
    "Install the required dependency:",
    "",
    "```bash",
    "pip install -r requirements.txt",
    "```",
    "",
    "The project was developed using Python 3.x, Ultralytics 8.4.163, PyTorch with CUDA support, and an NVIDIA Tesla T4 during training.",
    "",
    "---",
    "",
    "## 15. Results",
    "",
    "Important result files are available under:",
    "",
    "```text",
    "results/training/",
    "results/evaluation/",
    "results/predictions/",
    "```",
    "",
    "These include training curves, precision-recall curves, F1 curves, confusion matrices, validation predictions, and YOLOv8n versus YOLOv8s visual comparisons.",
    "",
    "---",
    "",
    "## 16. Conclusion",
    "",
    "This project demonstrates a YOLOv8-based approach for thermal anomaly detection and segmentation in solar-panel images.",
    "",
    "The workflow covers dataset preparation, taxonomy consolidation, segmentation annotation processing, model training, evaluation, error analysis, and controlled model comparison.",
    "",
    "The experiments also demonstrate practical challenges including class imbalance, limited examples for some fault categories, and the difficulty of precise anomaly segmentation.",
    "",
    "---",
    "",
    "## 17. References",
    "",
    "- Ultralytics YOLO",
    "- Thermal Anomalies — Solar Panel Data Set",
    "- Original dataset source used during project preparation",
    "",
    "> Add the exact dataset URL to this section before publishing the repository."
]

README.write_text("\n".join(lines) + "\n", encoding="utf-8")

print("=" * 80)
print("STAGE 10.11 COMPLETE")
print("=" * 80)
print("README created:")
print(README)
print(f"Size: {README.stat().st_size / 1024:.1f} KB")

In [ ]:
from pathlib import Path
import json

PROJECT = Path("/content/Solar-Panel-Thermal-Anomaly-Detection")

print("=" * 80)
print("STAGE 10.12 — FINAL GITHUB REPOSITORY AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# Required files
# ------------------------------------------------------------

required_files = [
    "README.md",
    "requirements.txt",
    "Solar_Panel_Thermal_Anomaly_Detection.ipynb",
    "models/yolov8n_seg_baseline_best.pt",
    "models/yolov8s_seg_improved_best.pt",
    "results/training/yolov8n_baseline_results.png",
    "results/training/yolov8s_improved_results.png",
    "results/evaluation/model_metrics.txt",
]

print("\nREQUIRED FILES")
print("-" * 80)

missing = []

for relative in required_files:
    path = PROJECT / relative

    if path.exists():
        size = path.stat().st_size
        print(f"✓ {relative}  ({size:,} bytes)")
    else:
        print(f"✗ MISSING: {relative}")
        missing.append(relative)

# ------------------------------------------------------------
# Check evaluation directory
# ------------------------------------------------------------

evaluation_dir = PROJECT / "results/evaluation/yolov8s_test"

evaluation_files = [
    "BoxF1_curve.png",
    "BoxPR_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
    "MaskF1_curve.png",
    "MaskPR_curve.png",
    "MaskP_curve.png",
    "MaskR_curve.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
]

print("\nYOLOv8s EVALUATION FILES")
print("-" * 80)

for filename in evaluation_files:
    path = evaluation_dir / filename

    if path.exists():
        print(f"✓ {filename}")
    else:
        print(f"✗ MISSING: {filename}")
        missing.append(f"results/evaluation/yolov8s_test/{filename}")

# ------------------------------------------------------------
# Check prediction directories
# ------------------------------------------------------------

print("\nMODEL COMPARISON PREDICTIONS")
print("-" * 80)

for model_name in ["yolov8n", "yolov8s"]:
    directory = PROJECT / "results/predictions/model_comparison" / model_name

    if directory.exists():
        images = list(directory.glob("*.jpg")) + list(directory.glob("*.png"))
        print(f"✓ {model_name}: {len(images)} prediction images")
    else:
        print(f"✗ Missing directory: {model_name}")
        missing.append(str(directory.relative_to(PROJECT)))

# ------------------------------------------------------------
# Validate README
# ------------------------------------------------------------

print("\nREADME CHECK")
print("-" * 80)

readme = PROJECT / "README.md"

if readme.exists():
    text = readme.read_text(encoding="utf-8")

    checks = [
        "# Solar Panel Thermal Anomaly Detection Using YOLOv8",
        "## 3. Dataset",
        "## 4. Final Taxonomy",
        "## 8. Baseline Test Results",
        "## 10. Improvement Experiment",
        "## 11. YOLOv8n vs YOLOv8s",
        "## 12. Limitations",
        "## 16. Conclusion",
    ]

    for item in checks:
        if item in text:
            print(f"✓ {item}")
        else:
            print(f"✗ Missing README section: {item}")

# ------------------------------------------------------------
# Validate notebook JSON
# ------------------------------------------------------------

print("\nNOTEBOOK CHECK")
print("-" * 80)

notebook = PROJECT / "Solar_Panel_Thermal_Anomaly_Detection.ipynb"

try:
    notebook_data = json.loads(
        notebook.read_text(encoding="utf-8")
    )

    print("✓ Valid JSON")
    print(f"✓ Notebook cells: {len(notebook_data.get('cells', []))}")
    print(f"✓ Notebook format: {notebook_data.get('nbformat')}")

except Exception as e:
    print("✗ Notebook validation failed")
    print(e)
    missing.append("Notebook JSON validation")

# ------------------------------------------------------------
# Final project file count
# ------------------------------------------------------------

all_files = [
    p for p in PROJECT.rglob("*")
    if p.is_file()
]

print("\nPROJECT SUMMARY")
print("-" * 80)
print(f"Total files: {len(all_files)}")

if missing:
    print("\n⚠ AUDIT FOUND MISSING ITEMS:")
    for item in missing:
        print(" -", item)
else:
    print("\n✓ ALL REQUIRED PROJECT FILES ARE PRESENT")

print("\n" + "=" * 80)

if not missing:
    print("STAGE 10.12 COMPLETE — REPOSITORY READY FOR FINAL REVIEW")
else:
    print("STAGE 10.12 COMPLETE — FIX THE ITEMS ABOVE BEFORE PUBLISHING")

print("=" * 80)

In [ ]:
from pathlib import Path

PROJECT = Path("/content/Solar-Panel-Thermal-Anomaly-Detection")
README = PROJECT / "README.md"

text = README.read_text(encoding="utf-8")

old = "> Add the exact dataset URL to this section before publishing the repository."

new = (
    "**Dataset source:** "
    "[Thermal Anomalies — Solar Panel Data Set]"
    "(https://universe.roboflow.com/solar-panel-data-set/thermal-anomalies-jev7w)"
)

if old in text:
    text = text.replace(old, new)
    README.write_text(text, encoding="utf-8")
    print("✓ Dataset source link added to README")
else:
    print("⚠ The placeholder was not found.")
    print("The README may already contain the dataset source.")

print("\nREADME:")
print(README)

In [ ]:
%pip install -q gradio

from pathlib import Path
from ultralytics import YOLO
import gradio as gr
import numpy as np

MODEL_PATH = Path(
    "/content/Solar-Panel-Thermal-Anomaly-Detection/"
    "models/yolov8s_seg_improved_best.pt"
)

print("=" * 80)
print("STAGE 11.3 — INTERACTIVE SOLAR PANEL PREDICTION DEMO")
print("=" * 80)

# Load the trained YOLOv8s segmentation model
model = YOLO(str(MODEL_PATH))

print("\n✓ YOLOv8s model loaded")
print("Model:", MODEL_PATH)
print("\n✓ Ready for image prediction")


def predict_image(image):
    """
    Run YOLOv8s segmentation on an uploaded image.
    Returns the annotated prediction image and prediction summary.
    """

    if image is None:
        return None, "Please upload a solar-panel thermal image."

    results = model.predict(
        source=image,
        imgsz=640,
        conf=0.25,
        verbose=False
    )

    result = results[0]

    # YOLO annotated image
    annotated = result.plot()

    # Convert BGR -> RGB
    annotated = annotated[:, :, ::-1]

    # Prediction summary
    if result.boxes is None or len(result.boxes) == 0:
        summary = "No anomaly detected at the selected confidence threshold."
    else:
        lines = []
        names = result.names

        for i in range(len(result.boxes)):
            class_id = int(result.boxes.cls[i].item())
            confidence = float(result.boxes.conf[i].item())
            class_name = names[class_id]

            lines.append(
                f"{i + 1}. {class_name} — "
                f"Confidence: {confidence:.2%}"
            )

        summary = "\n".join(lines)

    return annotated, summary


demo = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(
        type="numpy",
        label="Upload Solar Panel Thermal Image"
    ),
    outputs=[
        gr.Image(
            label="Detection & Segmentation Result"
        ),
        gr.Textbox(
            label="Detected Anomalies",
            lines=6
        )
    ],
    title="Solar Panel Thermal Anomaly Detector",
    description=(
        "Upload a thermal solar-panel image. "
        "The YOLOv8s segmentation model will detect and "
        "highlight thermal anomalies."
    ),
    examples=None
)

print("\n" + "=" * 80)
print("STAGE 11.3 COMPLETE")
print("=" * 80)
print("The interactive demo is ready.")
print("Run the Gradio interface below to upload an image.")
print("=" * 80)

demo.launch(share=True)

In [ ]:
from pathlib import Path
import random
from IPython.display import display
from PIL import Image

TEST_DIR = Path("/content/Solar_Thermal_Anomaly_Segmentation/test/images")

test_images = list(TEST_DIR.glob("*"))

sample_image = random.choice(test_images)

print("Selected test image:")
print(sample_image)

display(Image.open(sample_image))

In [ ]:
from pathlib import Path
import random
import zipfile
from google.colab import files

# ============================================================
# CREATE ZIP OF 20 RANDOM TEST IMAGES
# ============================================================

TEST_DIR = Path("/content/Solar_Thermal_Anomaly_Segmentation/test/images")
OUTPUT_ZIP = Path("/content/20_thermal_test_images.zip")

# Find image files
test_images = [
    p for p in TEST_DIR.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
]

print("Total test images found:", len(test_images))

# Make sure there are at least 20
if len(test_images) < 20:
    raise ValueError(
        f"Only {len(test_images)} images found. "
        "Need at least 20 images."
    )

# Select 20 random images
sample_images = random.sample(test_images, 20)

print("\nSelected images:")
for i, image_path in enumerate(sample_images, 1):
    print(f"{i:02d}. {image_path.name}")

# Create ZIP
with zipfile.ZipFile(
    OUTPUT_ZIP,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    for image_path in sample_images:
        z.write(
            image_path,
            arcname=image_path.name
        )

print("\n" + "=" * 50)
print("✅ ZIP CREATED SUCCESSFULLY")
print("=" * 50)
print("File:", OUTPUT_ZIP)
print("Images:", len(sample_images))
print("Size:", round(OUTPUT_ZIP.stat().st_size / 1024, 2), "KB")

# Download to your computer
files.download(str(OUTPUT_ZIP))

## Final Results, Visualizations, and References

This final section prepares submission-ready outputs **from files created by your actual Colab run**. Run it after dataset preparation and both model-training stages. If a model checkpoint is missing, the cell will report that instead of inventing metrics.

### Dataset source and download details
- **Dataset:** Thermal Anomalies — Solar Panel Data Set (Roboflow Universe)
- **Dataset page:** https://universe.roboflow.com/solar-panel-data-set/thermal-anomalies-jev7w
- **Version used in this project:** Version 2
- **Dataset splits recorded during verification:** 711 training, 202 validation, and 100 test images (1,013 total).
- **Original annotation taxonomy:** 17 classes. The project notebook prepares a reduced five-class segmentation dataset for the experiments; refer to the class-mapping cells and generated `data.yaml` for the exact mapping used in your run.

### Results and visualizations generated below
The next cell searches for training curves, confusion matrices, and model checkpoints. Where the trained checkpoints and prepared dataset are available, it evaluates the models on the test split, saves a metrics CSV, and generates prediction overlays for test images. These are the outputs to include in your report/poster.

### References
1. Roboflow Universe. *Thermal Anomalies — Solar Panel Data Set*. https://universe.roboflow.com/solar-panel-data-set/thermal-anomalies-jev7w
2. Ultralytics. *YOLOv8 Documentation*. https://docs.ultralytics.com/models/yolov8/
3. Ultralytics. *Instance Segmentation Task Documentation*. https://docs.ultralytics.com/tasks/segment/
4. Ultralytics. *Model Validation Documentation*. https://docs.ultralytics.com/modes/val
5. Google. *Google Colaboratory*. https://colab.research.google.com/

**Citation note:** cite the dataset page and software/documentation in your submission. Report the metrics produced by this notebook run; do not present planned or previously copied numbers as fresh results unless they are reproduced by the saved evaluation outputs.


In [ ]:
# FINAL OUTPUT CELL — generate actual results, figures, and exportable files
from pathlib import Path
import shutil
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from PIL import Image as PILImage
from ultralytics import YOLO
import random

# Locate the prepared dataset and output directories.
DATA_CANDIDATES = [
    Path("/content/Solar_Thermal_Anomaly_Segmentation/data.yaml"),
    Path("/content/Thermal-Anomalies-2/data.yaml"),
]
DATA_YAML = next((p for p in DATA_CANDIDATES if p.exists()), None)

RUNS = Path("/content/solar_panel_runs")
PROJECT = Path("/content/Solar-Panel-Thermal-Anomaly-Detection")
RESULTS_DIR = PROJECT / "results"
TRAINING_DIR = RESULTS_DIR / "training"
EVAL_DIR = RESULTS_DIR / "evaluation"
PRED_DIR = RESULTS_DIR / "predictions"
for folder in [TRAINING_DIR, EVAL_DIR, PRED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

MODEL_PATHS = {
    "YOLOv8n-seg": RUNS / "yolov8n_seg_baseline/weights/best.pt",
    "YOLOv8s-seg": RUNS / "yolov8s_seg_improved/weights/best.pt",
}

print("=" * 80)
print("FINAL RESULTS AND VISUALIZATION EXPORT")
print("=" * 80)
print("Prepared dataset YAML:", DATA_YAML if DATA_YAML else "NOT FOUND")
print("\nModel checkpoints:")
for label, p in MODEL_PATHS.items():
    print(f"  {label}: {'FOUND' if p.exists() else 'NOT FOUND'} — {p}")

# 1) Copy actual training curves and any existing evaluation plots into the project.
copied = []
for source in RUNS.rglob("*"):
    if not source.is_file():
        continue
    lower = source.name.lower()
    if lower in {"results.png", "confusion_matrix.png", "confusion_matrix_normalized.png",
                 "pr_curve.png", "f1_curve.png", "p_curve.png", "r_curve.png"}:
        if "yolov8n_seg_baseline" in str(source):
            tag = "yolov8n_baseline"
        elif "yolov8s_seg_improved" in str(source):
            tag = "yolov8s_improved"
        else:
            tag = source.parent.name
        dest_dir = TRAINING_DIR if lower == "results.png" else EVAL_DIR
        dest = dest_dir / f"{tag}_{source.name}"
        if source.resolve() != dest.resolve():
            shutil.copy2(source, dest)
        copied.append(dest)

print(f"\nCopied {len(copied)} existing training/evaluation plot(s).")

# 2) Evaluate any available trained model on the test split and save measured metrics.
metric_rows = []
loaded_models = {}
if DATA_YAML:
    for label, checkpoint in MODEL_PATHS.items():
        if not checkpoint.exists():
            continue
        print(f"\nEvaluating {label} on test split...")
        model = YOLO(str(checkpoint))
        loaded_models[label] = model
        try:
            metrics = model.val(
                data=str(DATA_YAML),
                split="test",
                imgsz=640,
                plots=True,
                project=str(EVAL_DIR),
                name=label.lower().replace("-", "_").replace(" ", "_") + "_test",
                exist_ok=True,
                device=0 if __import__("torch").cuda.is_available() else "cpu",
                verbose=False
            )
            row = {"Model": label}
            for prefix, group in [("Box", getattr(metrics, "box", None)),
                                  ("Mask", getattr(metrics, "seg", None))]:
                if group is not None:
                    row[f"{prefix} Precision"] = float(getattr(group, "mp", float("nan")))
                    row[f"{prefix} Recall"] = float(getattr(group, "mr", float("nan")))
                    row[f"{prefix} mAP50"] = float(getattr(group, "map50", float("nan")))
                    row[f"{prefix} mAP50-95"] = float(getattr(group, "map", float("nan")))
            metric_rows.append(row)
            print("Test evaluation complete.")
        except Exception as exc:
            print(f"Evaluation failed for {label}: {type(exc).__name__}: {exc}")
else:
    print("\nCannot run fresh evaluation: prepared data.yaml was not found.")

if metric_rows:
    metrics_df = pd.DataFrame(metric_rows)
    metrics_csv = EVAL_DIR / "test_metrics_actual_run.csv"
    metrics_df.to_csv(metrics_csv, index=False)
    print("\nActual test metrics from this run:")
    display(metrics_df.round(4))
    print("Saved:", metrics_csv)
else:
    print("\nNo fresh metric table was created. Train the models and ensure data.yaml is available, then rerun this cell.")

# 3) Display and save a comparison chart when fresh metrics are available.
if metric_rows:
    chart_df = pd.DataFrame(metric_rows).set_index("Model")
    columns = [c for c in chart_df.columns if c.endswith("mAP50") or c.endswith("mAP50-95")]
    if columns:
        ax = chart_df[columns].plot(kind="bar", figsize=(10, 5))
        ax.set_title("YOLOv8 Segmentation Model Test-Set Comparison")
        ax.set_ylabel("Score")
        ax.set_ylim(0, 1)
        ax.set_xlabel("")
        plt.xticks(rotation=0)
        plt.tight_layout()
        chart_path = EVAL_DIR / "test_metrics_comparison.png"
        plt.savefig(chart_path, dpi=180, bbox_inches="tight")
        plt.show()
        print("Saved:", chart_path)

# 4) Show available training curves and confusion matrices.
plot_paths = sorted(set(
    list(TRAINING_DIR.glob("*results.png")) +
    list(EVAL_DIR.rglob("*confusion_matrix*.png")) +
    list(EVAL_DIR.rglob("*results.png"))
))
if plot_paths:
    print("\nTraining curves / confusion matrices:")
    for plot_path in plot_paths[:8]:
        print(plot_path.name)
        display(Image(filename=str(plot_path), width=850))
else:
    print("\nNo saved training/evaluation plots were found yet. Run the training and evaluation cells first.")

# 5) Save prediction overlays for a few test images, if checkpoints and test images are present.
TEST_CANDIDATES = [
    Path("/content/Solar_Thermal_Anomaly_Segmentation/test/images"),
    Path("/content/Thermal-Anomalies-2/test/images"),
]
TEST_DIR = next((p for p in TEST_CANDIDATES if p.exists()), None)
test_images = []
if TEST_DIR:
    test_images = sorted([p for p in TEST_DIR.iterdir()
                          if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}])
if loaded_models and test_images:
    random.seed(42)
    sample_images = random.sample(test_images, min(4, len(test_images)))
    for label, model in loaded_models.items():
        label_dir = PRED_DIR / label.lower().replace("-", "_").replace(" ", "_")
        label_dir.mkdir(parents=True, exist_ok=True)
        print(f"\nSample predictions — {label}")
        for image_path in sample_images:
            try:
                pred = model.predict(
                    source=str(image_path),
                    imgsz=640,
                    conf=0.25,
                    save=True,
                    project=str(label_dir),
                    name="overlays",
                    exist_ok=True,
                    device=0 if __import__("torch").cuda.is_available() else "cpu",
                    verbose=False
                )
                saved = getattr(pred[0], "save_dir", None)
                if saved:
                    saved_image = Path(saved) / image_path.name
                    if saved_image.exists():
                        print(f"{image_path.name} → {saved_image}")
                        display(Image(filename=str(saved_image), width=700))
            except Exception as exc:
                print(f"Prediction failed for {image_path.name}: {exc}")
else:
    print("\nPrediction overlays not generated. Confirm test images and at least one trained checkpoint exist.")

# 6) Write reference list into the repository and print expected output locations.
references = """# References

1. Roboflow Universe, Thermal Anomalies — Solar Panel Data Set:
   https://universe.roboflow.com/solar-panel-data-set/thermal-anomalies-jev7w
2. Ultralytics, YOLOv8 Documentation:
   https://docs.ultralytics.com/models/yolov8/
3. Ultralytics, Instance Segmentation Documentation:
   https://docs.ultralytics.com/tasks/segment/
4. Ultralytics, Model Validation Documentation:
   https://docs.ultralytics.com/modes/val
5. Google Colaboratory:
   https://colab.research.google.com/
"""
(PROJECT / "references").mkdir(parents=True, exist_ok=True)
(PROJECT / "references" / "REFERENCES.md").write_text(references, encoding="utf-8")

print("\n" + "=" * 80)
print("OUTPUT LOCATIONS")
print("=" * 80)
print("Training plots :", TRAINING_DIR)
print("Evaluation     :", EVAL_DIR)
print("Predictions    :", PRED_DIR)
print("References     :", PROJECT / "references" / "REFERENCES.md")
print("\nNote: outputs appear only for stages that actually ran successfully.")
